# 20 — Avaliação final e encerramento experimental

Esta é a última etapa experimental desta linha do TCC. Avalia **baseline e
candidato 10 nos mesmos 300 IDs da reserva externa**, seguindo o protocolo
fixado no notebook 19 antes da consulta à reserva.

- Confiança dos alertas **0,25**, política **V2** e IoU de associação **0,5**.
- Captura em confiança 0,001; nenhum ajuste de pesos ou limiares.
- Precisão, recall, F1, omissões altas com denominadores e resultados por classe.
- ICs de 95% por **2.000 reamostragens pareadas de grupos de imagens**.
- Relatório, tabelas e texto-base para o relato do TCC.
- Retomada das previsões já concluídas e envio automático dos resultados ao Drive.

Não exige formulários, seleção de imagens ou edição de parâmetros. Home TEST e
Open Images TEST 20k ficam fora desta avaliação. A reserva é final para este
fluxo documentado; o pré-treino completo do baseline permanece não comprovado.


## 1. Preparar o ambiente e o código

Importe este arquivo como **novo notebook no Kaggle**. Ative **Internet** e
habilite o Secret existente **GDRIVE_RCLONE_CONFIG_B64** em **Add-ons → Secrets**.
GPU **T4** é recomendada. Se não houver GPU disponível, o código usa CPU
automaticamente, mantendo os parâmetros da avaliação.

Use **Save Version → Save & Run All**. Não copie células de outros notebooks.

**Estimativa: 5–15 minutos com T4; 10–30 minutos em CPU**, incluindo downloads e
salvamentos usuais. É uma estimativa de planejamento; conexão e disponibilidade
do Kaggle podem ampliar o tempo. Não há treinamento.

Ao término, os resultados estão em **pacote_avaliacao_final_20.zip**.


In [ ]:
%pip install -q ultralytics==8.4.150


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
os.environ['YOLO_CONFIG_DIR']='/tmp/tcc_yolo20_config'
os.environ['WANDB_DISABLED']='true'
import sys,json,hashlib,platform,shutil,time,random
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import PIL,torch,ultralytics
from IPython.display import display,FileLink
if ultralytics.__version__!='8.4.150':
    raise RuntimeError('A versão importada do Ultralytics difere da fixada. Reinicie a sessão e use Run All.')
DEVICE=0 if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False
random.seed(19002026);np.random.seed(19002026);torch.manual_seed(19002026)
AMBIENTE={'python':platform.python_version(),'numpy':np.__version__,'pillow':PIL.__version__,
    'torch':torch.__version__,'ultralytics':ultralytics.__version__,'cuda_runtime':torch.version.cuda,
    'dispositivo':str(DEVICE),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'half':False,'tf32':False}
BASE_ID='fab456a7e1eb48881fef07c5'

PASTA_LOCAL=Path('/tmp/tcc_avaliacao_final_20')/BASE_ID
PASTA_BOOT=Path('/kaggle/working/resultados_tcc')/('preparacao20_'+BASE_ID)
PASTA_LOCAL.mkdir(parents=True,exist_ok=True);PASTA_BOOT.mkdir(parents=True,exist_ok=True)
PASTA_RESULTADOS=PASTA_BOOT;NOME_EXPERIMENTO=PASTA_BOOT.name;DRIVE=None
INICIO=time.monotonic();ULTIMA_ETAPA=INICIO;TEMPOS=[]
print('Dispositivo:',AMBIENTE['gpu'] or 'CPU')
print('Dois modelos fixos, 300 imagens por modelo. Nenhum treinamento.')

EXPECTED={'pacote_19': '33b9937613b3cddae984a4499adc2788318cb957628c858d525bc2a86be674e4', 'contexto_19': '4d5ea2227cb3d7146cc78d8b5d752047cad25b1121335d051e7750dc79098db6', 'pasta_19': 'resultados/consolidacao19_4d5ea2227cb3d7146cc78d8b', 'protocolo_sha256': 'b13f47fe8c2bc87369ccba1f5dbdad8d4fff46651531956617d38684e0b61b38', 'weights': {'baseline': 'modelos/yolov8n_baseline/best.pt', 'candidato_10': 'resultados/yolov8n_home_openimages_80_20_1ep_kaggle_20261015_030228_068010/best.pt'}}
PROTOCOLO_CONGELADO={'version': '20-plano-avaliacao-final-congelado-em-19-v1', 'status': 'planejado; reserva nao consultada no 19', 'primary_model': 'baseline', 'comparison_model': 'candidato_10', 'weight_sha256': {'baseline': '4170653e0c0a604f85b6747e42b3fd2ff5ddb765ee4762915dfc9f3d33d322d9', 'candidato_10': '97040f91876d8835a67a7db287d3e1c5f129a4068e36dd4483d947f62b260c5f'}, 'dataset_sha256': '695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0', 'archive_sha256': '5af596546e2ad734c7b663f4567de2087d7fb42b41527d4852affdb4e7d2b96c', 'dataset_location': 'datasets/validacao_independente_11_v1/695b445e15c905642e2b276801b71132ced28e5eaee92d9fbcdd608f582f15a0/validacao_independente_11_695b445e15c9.zip', 'role': 'reserva_futura', 'images': 300, 'population': 'todos os 300 IDs do papel reserva_futura no manifesto autenticado; sem substituicoes', 'classes': 'mesmo mapeamento de 12 classes; cobertura parcial original preservada', 'policy_sha256': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'matcher_sha256': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d', 'inference': {'ultralytics': '8.4.150', 'conf': 0.001, 'imgsz': 640, 'rect': True, 'iou': 0.7, 'max_det': 300, 'half': False, 'batch': 1}, 'alert_confidence': 0.25, 'match_iou': 0.5, 'policy': 'v2', 'primary_estimands': ['recall de alertas por objeto do baseline', 'delta recall candidato10 menos baseline'], 'secondary': ['precisao', 'F1', 'omissoes altas: contagem e taxa por objetos GT de prioridade alta', 'metricas por classe e suporte', 'contagens TP/FN/FP e denominadores'], 'confidence_intervals': {'method': 'bootstrap pareado de grupos de imagens; manter bytes iguais e pHash64 distancia<=4 no mesmo grupo por ligacao transitiva', 'repetitions': 2000, 'seed': 19002026, 'confidence': 0.95, 'interval': 'percentil 2.5 e 97.5; grupos reamostrados com reposicao; mesmos grupos para ambos os modelos', 'zero_denominator': 'replica sem suporte fica indefinida; informar quantidade de replicas validas por metrica'}, 'low_support': 'informar contagens; sinalizar classe com menos de 30 GT, sem ocultar resultados', 'weight_or_threshold_tuning_after_reserve': False, 'second_choice_after_failure': False, 'deployment_decision': 'nenhuma troca automatica; caracterizacao dos dois modelos predefinidos', 'historical_rules': 'os seis limites das validacoes ficam no desenvolvimento; nao aplicar limites absolutos de 1.000 imagens em 300', 'home_test': 'fora desta avaliacao primaria; historico original do baseline nao comprovado', 'openimages_test20k': 'fora desta etapa', 'training': False, 'limits': ['A independencia refere-se a reserva preservada no fluxo documentado; nao comprova ausencia de exposicao em todo pretreino original.', 'Nao valida distancia, risco fisico ou uso assistivo em ambiente real.']}
MODULOS_EMBUTIDOS={'conf14_data.py': '"""Entradas autenticadas do diagnóstico 14; extrai somente Home VAL."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nfrom PIL import Image\nimport diag11_core as core\n\nCOHORTS = (\'home_val_historica\', \'openimages_val_nova\')\nBOOLS = {\'True\':True, \'False\':False}\n\ndef rows(raw):\n    return list(csv.DictReader(io.StringIO(raw.decode(\'utf-8-sig\'))))\n\ndef parse_events(raw):\n    result = rows(raw)\n    for r in result:\n        r[\'classe_id\'] = int(r[\'classe_id\'])\n        for field in [\'bbox_ref\',\'bbox_pred\']:\n            r[field] = json.loads(r[field]) if r[field] else None\n        for field in [\'ha_alerta_ref\',\'ha_alerta_pred\']:\n            r[field] = BOOLS[r[field]]\n        r[\'confianca\'] = float(r[\'confianca\']) if r[\'confianca\'] else None\n    return result\n\ndef gt_key(r):\n    return (r[\'id\'],int(r[\'classe_id\']),tuple(round(float(v),6) for v in r[\'bbox_ref\']))\n\ndef gt_map(events):\n    result = {}\n    for r in events:\n        if r[\'bbox_ref\'] is None: continue\n        key = gt_key(r)\n        if key in result: raise ValueError(\'Objeto GT repetido nos registros.\')\n        result[key] = r\n    return result\n\ndef load_source12(path, expected, output=None):\n    if core.sha256_file(path)!=expected[\'pacote_12\']:\n        raise RuntimeError(\'O pacote 12 mudou em relação ao recebido.\')\n    bundle = {\'evaluations\':{}}\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())):\n            raise RuntimeError(\'Pacote 12 com membros repetidos.\')\n        for name in z.namelist(): core.safe_member(name)\n        if z.testzip() is not None: raise RuntimeError(\'Falha de CRC no pacote 12.\')\n        cfg = json.loads(z.read(\'configuracao_experimento_12.json\'))\n        summary = json.loads(z.read(\'resumo_experimento_12.json\'))\n        if not summary[\'concluido\'] or summary[\'reserva_futura_inferida\'] or summary[\'test_home_inferido\'] or summary[\'test_oi20k_inferido\']:\n            raise RuntimeError(\'Execução 12 incompatível com o protocolo.\')\n        if cfg[\'expected\'][\'candidato\']!=expected[\'candidato\'] or cfg[\'expected\'][\'politica\']!=expected[\'politica\'] or summary[\'dataset_sha256\']!=expected[\'dataset_novo\']:\n            raise RuntimeError(\'Referências ou política diferentes do experimento 12.\')\n        if cfg[\'codigo_embutido_sha256\']!=expected[\'codigo_12\']:\n            raise RuntimeError(\'Código do experimento 12 diferente do recebido.\')\n        for name, digest in expected[\'codigo_12\'].items():\n            if hashlib.sha256(z.read(\'codigo/\'+name)).hexdigest()!=digest:\n                raise RuntimeError(\'Código alterado no pacote 12: \'+name)\n        rule_raw = z.read(\'regra_selecao_congelada.json\')\n        if hashlib.sha256(rule_raw).hexdigest()!=cfg[\'regra_sha256\']:\n            raise RuntimeError(\'A regra original do experimento 12 foi alterada.\')\n        selection = json.loads(z.read(\'selecao_checkpoints.json\'))\n        checkpoint = next(e for e in selection[\'assessed\'] if e[\'id\']==\'B_home_replay_e03\')\n        if checkpoint[\'weight_sha256\']!=expected[\'B_e03\']:\n            raise RuntimeError(\'Checkpoint B_e03 diferente do esperado.\')\n        for model,prefix in [(\'baseline\',\'referencias/baseline\'),(\'candidato_10\',\'referencias/candidato_10\'),(\'B_e03\',\'avaliacoes/B_home_replay_e03\')]:\n            bundle[\'evaluations\'][model] = {}\n            for cohort in COHORTS:\n                base=prefix+\'/\'+cohort+\'/\'\n                members={name:z.read(base+name) for name in [\'metricas.json\',\'alertas_por_imagem.csv\',\'avaliacao_objetos.csv\']}\n                metrics=json.loads(members[\'metricas.json\'])\n                images=rows(members[\'alertas_por_imagem.csv\'])\n                for r in images:\n                    for key in [\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']:\n                        r[key]=int(r[key])\n                if len({r[\'id\'] for r in images})!=len(images) or len(images)!=metrics[\'imagens\']:\n                    raise RuntimeError(\'IDs ou quantidades inconsistentes nos registros 12.\')\n                for field, n in [(\'tp_alerta\',metrics[\'alertas_por_objeto\'][\'tp\']),(\'fn_alerta\',metrics[\'alertas_por_objeto\'][\'fn\']),(\'fp_alerta\',metrics[\'alertas_por_objeto\'][\'fp\']),(\'alta_ref\',metrics[\'alta_ref\']),(\'alta_sem_alerta\',metrics[\'alta_sem_alerta\'])]:\n                    if sum(r[field] for r in images)!=n: raise RuntimeError(\'Contagens inconsistentes no pacote 12.\')\n                events=parse_events(members[\'avaliacao_objetos.csv\'])\n                gt_map(events)\n                bundle[\'evaluations\'][model][cohort]={\'summary\':metrics,\'images\':images,\'events\':events}\n                if output is not None:\n                    dest=Path(output)/\'referencias_12\'/model/cohort\n                    dest.mkdir(parents=True,exist_ok=True)\n                    for name, raw in members.items(): (dest/name).write_bytes(raw)\n        for cohort in COHORTS:\n            target=gt_map(bundle[\'evaluations\'][\'B_e03\'][cohort][\'events\'])\n            for model in [\'baseline\',\'candidato_10\']:\n                source=gt_map(bundle[\'evaluations\'][model][cohort][\'events\'])\n                if source.keys()!=target.keys(): raise RuntimeError(\'Modelos do 12 usam referências GT diferentes.\')\n                if any(source[k][\'ha_alerta_ref\']!=target[k][\'ha_alerta_ref\'] or source[k][\'prioridade_ref\']!=target[k][\'prioridade_ref\'] for k in target):\n                    raise RuntimeError(\'Referências de prioridade diferentes entre modelos.\')\n        bundle.update(config=cfg,summary=summary,rule=json.loads(rule_raw),\n                      audit=json.loads(z.read(\'auditoria_dados.json\')))\n    return bundle\n\ndef extract_home_val(path, destination, expected_zip, expected_test, expected_policy):\n    if core.sha256_file(path)!=expected_zip: raise RuntimeError(\'ZIP Home alterado.\')\n    destination=Path(destination)\n    records=[];frozen=[]\n    with zipfile.ZipFile(path) as z:\n        if len(z.namelist())!=len(set(z.namelist())): raise RuntimeError(\'Membros repetidos no Home.\')\n        for name in z.namelist():core.safe_member(name)\n        manifest=json.loads(z.read(\'verificacao_preparacao.json\'))\n        if (manifest[\'test\'][\'sha256\']!=expected_test or manifest[\'test\'][\'imagens\']!=403 or\n            {int(k):v for k,v in manifest[\'classes\'].items()}!=dict(enumerate(core.NAMES))):\n            raise RuntimeError(\'Manifesto Home incompatível.\')\n        root=manifest[\'dataset_dir\']\n        items={r[\'arquivo\']:r for r in manifest[\'arquivos\']}\n        if len(items)!=len(manifest[\'arquivos\']): raise RuntimeError(\'Manifesto Home repetido.\')\n        train_hash=set();val_hash=set();all_hash=set();counts=Counter()\n        for name,item in items.items():\n            is_image=name.startswith(root+\'/images/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}\n            if not is_image:continue\n            all_hash.add(item[\'sha256\'])\n            if name.startswith(root+\'/images/train/raw_train/\'):\n                counts[\'train\']+=1;train_hash.add(item[\'sha256\'])\n            elif name.startswith(root+\'/images/val/raw_train/\'):\n                counts[\'val\']+=1;val_hash.add(item[\'sha256\'])\n        if counts!={\'train\':1824,\'val\':456} or train_hash&val_hash:\n            raise RuntimeError(\'A divisão atual de Home mudou ou tem bytes repetidos entre TRAIN e VAL.\')\n        for name in z.namelist():\n            if not(name.startswith(\'codigo/\') and name.endswith(\'.py\')):continue\n            raw=z.read(name)\n            if name in items and hashlib.sha256(raw).hexdigest()!=items[name][\'sha256\']:\n                raise RuntimeError(\'Código Home alterado.\')\n            out=destination/name;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n            frozen.append((out,hashlib.sha256(raw).hexdigest()))\n        for name,item in sorted(items.items()):\n            if not(name.startswith(root+\'/images/val/raw_train/\') and Path(name).suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}):continue\n            label=name.replace(\'/images/\',\'/labels/\').rsplit(\'.\',1)[0]+\'.txt\'\n            if label not in items:raise RuntimeError(\'Label Home ausente.\')\n            paths={}\n            for field,member in [(\'image\',name),(\'label\',label)]:\n                raw=z.read(member)\n                if hashlib.sha256(raw).hexdigest()!=items[member][\'sha256\']:raise RuntimeError(\'Home alterado: \'+member)\n                out=destination/member;out.parent.mkdir(parents=True,exist_ok=True);out.write_bytes(raw)\n                paths[field]=out;frozen.append((out,items[member][\'sha256\']))\n            with Image.open(paths[\'image\']) as im:\n                width,height=im.size;im.verify()\n            records.append({\'id\':\'home:\'+name,\'name\':name.split(\'/raw_train/\')[-1],**paths,\n                \'role\':\'home_val\',\'coorte\':\'home_val_historica\',\'width\':width,\'height\':height,\n                \'known_mask\':core.ALL_KNOWN,\'sha256\':item[\'sha256\'],\'label_sha256\':items[label][\'sha256\'],\n                \'gt\':core.parse_yolo(paths[\'label\'].read_bytes(),width,height,source=label)})\n    if len(records)!=456 or core.sha256_file(destination/\'codigo/src/politica_alertas.py\')!=expected_policy:\n        raise RuntimeError(\'Quantidade Home VAL ou política V2 incompatível.\')\n    audit={\'home_train_manifesto\':1824,\'home_val_extraida\':456,\'home_train_extraida\':False,\n           \'home_test_extraido\':False,\'test_home_manifesto\':403,\'intersecao_bytes_train_val\':0,\n           \'historico_original_baseline_nao_comprovado_pela_separacao_atual\':True}\n    return records,all_hash,frozen,audit\n\ndef validate_cohorts(cohorts,bundle,reserved_ids):\n    if set(cohorts)!=set(COHORTS) or {k:len(v) for k,v in cohorts.items()}!={\'home_val_historica\':456,\'openimages_val_nova\':1000}:\n        raise ValueError(\'Coortes diferentes das duas validações congeladas.\')\n    for cohort,records in cohorts.items():\n        ids={r[\'id\'] for r in records}\n        if len(ids)!=len(records) or ids&set(reserved_ids):raise ValueError(\'ID repetido ou reservado na avaliação.\')\n        expected_role=\'home_val\' if cohort==\'home_val_historica\' else \'val_nova\'\n        if any(r[\'role\']!=expected_role for r in records):raise ValueError(\'Papel incompatível com a coorte de validação.\')\n        old=bundle[\'evaluations\'][\'B_e03\'][cohort]\n        if ids!={r[\'id\'] for r in old[\'images\']}:raise ValueError(\'IDs diferentes dos avaliados no 12.\')\n        actual={}\n        for r in records:\n            for g in r[\'gt\']:\n                if r[\'known_mask\']&(1<<g[\'classe_id\']):\n                    k=(r[\'id\'],g[\'classe_id\'],tuple(round(float(x),6) for x in g[\'bbox\']))\n                    if k in actual:raise ValueError(\'GT repetido na entrada atual.\')\n                    actual[k]=True\n        if actual.keys()!=gt_map(old[\'events\']).keys():raise ValueError(\'Caixas avaliáveis diferentes das referências originais.\')\n    return True\n', 'conf14_evaluation.py': '"""Varredura de confiança: uma inferência por imagem, política V2 fixa."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_data as data\nfrom exp12_protocol import judge\n\nTHRESHOLDS=(.05,.10,.15,.20,.25,.30,.40)\nPREDICT_ARGS={\'conf\':.001,\'imgsz\':640,\'rect\':True,\'iou\':.70,\'max_det\':300,\'half\':False,\'batch\':1}\nVERSION=\'14-confianca-automatica-1.0\'\n\ndef point_name(conf):return \'conf_\'+f\'{conf:.2f}\'.replace(\'.\',\'\')\n\n\ndef cache_request(records,weight_sha,environment):\n    if not records or len({r[\'id\'] for r in records})!=len(records):raise ValueError(\'Cache exige IDs únicos.\')\n    if any(r[\'role\'] not in {\'home_val\',\'val_nova\'} for r in records):raise ValueError(\'Cache somente para validação.\')\n    signature=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':VERSION,\'peso_sha256\':weight_sha,\'parametros\':PREDICT_ARGS,\n            \'ambiente\':environment,\'coorte\':records[0][\'coorte\'],\'imagens\':len(records),\n            \'entradas_sha256\':hashlib.sha256(core.canonical(signature).encode()).hexdigest()}\n\ndef read_cache(path,records):\n    expected={r[\'id\']:r for r in records};result={}\n    with gzip.open(path,\'rt\',encoding=\'utf-8\') as f:\n        for line in f:\n            row=json.loads(line);iid=row[\'id\']\n            if iid not in expected or iid in result:raise ValueError(\'Cache com ID desconhecido ou repetido.\')\n            r=expected[iid]\n            if row[\'image_sha256\']!=r[\'sha256\'] or row[\'label_sha256\']!=r[\'label_sha256\']:\n                raise ValueError(\'Cache pertence a outros bytes de imagem/label.\')\n            validate_predictions(row[\'predicoes\'],r);result[iid]=row[\'predicoes\']\n    if result.keys()!=expected.keys():raise ValueError(\'Cache incompleto.\')\n    return result\n\n\ndef event(record,kind,g,p,overlap,pred_low,conf):\n    row=core.record_event(record,\'B_e03\',kind,g,p,overlap,pred_low)\n    row[\'limiar_confianca\']=conf\n    row[\'melhor_confianca_mesma_classe\']=None\n    if kind==\'objeto_nao_detectado\':\n        flags=[v for v in row[\'indicios\'].split(\'|\') if v and v not in {\'deteccao_abaixo_da_confianca_025\',\'sobreposicao_com_previsao_de_outra_classe\',\'omissao_sem_indicio_automatico\'}]\n        matrix=core.iou_matrix([g],pred_low)[0]\n        same=[i for i,p0 in enumerate(pred_low) if p0[\'classe_id\']==g[\'classe_id\']]\n        if same:\n            best=max(same,key=lambda i:matrix[i]);row[\'melhor_confianca_mesma_classe\']=pred_low[best][\'confianca\']\n            if matrix[best]>=.5 and pred_low[best][\'confianca\']<conf:flags.append(\'deteccao_abaixo_do_limiar_avaliado\')\n        if any(matrix[i]>=.5 and p0[\'classe_id\']!=g[\'classe_id\'] and p0[\'confianca\']>=conf for i,p0 in enumerate(pred_low)):\n            flags.append(\'sobreposicao_com_previsao_de_outra_classe\')\n        row[\'indicios\']=\'|\'.join(flags or [\'omissao_sem_indicio_automatico\'])\n    return row\n\ndef analyse(record,pred_low,conf,policy,matcher):\n    if record[\'role\'] not in {\'home_val\',\'val_nova\'} or conf not in THRESHOLDS:\n        raise ValueError(\'Papel ou limiar fora do protocolo.\')\n    known=int(record[\'known_mask\'])\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in pred_low if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in pred_low if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[event(record,\'correspondencia\',gt[gi],pred[pi],float(iou),low,conf) for gi,pi,iou in pairs]\n    events += [event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    messages=[policy.gerar_mensagem(p,\'medio\') for p in all_pred if core.alert_on(p)]\n    if any(not isinstance(s,str) or not s.strip() for s in messages):raise RuntimeError(\'Alerta sem mensagem.\')\n    im={\'coorte\':record[\'coorte\'],\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':\'B_e03\',\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return im,events\n\ndef summarise(images,events,conf):\n    if not images:raise ValueError(\'Validação vazia.\')\n    total=lambda k:sum(r[k] for r in images)\n    complete=[r for r in images if r[\'cobertura_completa_alertas\']]\n    summary={\'modelo\':\'B_e03\',\'coorte\':images[0][\'coorte\'],\'imagens\':len(images),\'conf_alertas\':conf,\n        \'alertas_por_objeto\':core.alert_metrics(total(\'tp_alerta\'),total(\'fn_alerta\'),total(\'fp_alerta\')),\n        \'alta_ref\':total(\'alta_ref\'),\'alta_sem_alerta\':total(\'alta_sem_alerta\'),\n        \'taxa_alta_sem_alerta\':core.ratio(total(\'alta_sem_alerta\'),total(\'alta_ref\')),\n        \'detector_no_limiar\':core.alert_metrics(total(\'tp_detector\'),total(\'fn_detector\'),total(\'fp_detector\')),\n        \'gt_ignorados_sem_cobertura\':total(\'gt_ignorados\'),\'predicoes_ignoradas_conf\':total(\'predicoes_ignoradas_conf\'),\n        \'imagens_com_cobertura_completa_alertas\':len(complete),\n        \'alertas_por_imagem_cobertura_completa\':core.alert_metrics(\n            sum(r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete),\n            sum(r[\'ha_alerta_ref\'] and not r[\'ha_alerta_pred\'] for r in complete),\n            sum(not r[\'ha_alerta_ref\'] and r[\'ha_alerta_pred\'] for r in complete)),\n        \'referencia\':\'Concordância com política V2 nas caixas; não valida risco físico.\',\n        \'nao_e_openimages_challenge\':True,\'ap_recalculada\':False}\n    classes=[]\n    for cid,name in enumerate(core.NAMES):\n        part=[e for e in events if e[\'classe_id\']==cid];n=sum(e[\'bbox_ref\']!=\'\' for e in part)\n        classes.append({\'classe_id\':cid,\'classe\':name,\'gt_avaliaveis\':n,\'suporte_reduzido\':n<30,\'conf_alertas\':conf,\n            **core.alert_metrics(sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part),\n                                 sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in part),\n                                 sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in part)),\n            \'alta_ref\':sum(e[\'prioridade_ref\']==\'alto\' for e in part),\n            \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in part)})\n    return summary,classes\n\ndef evaluate_cache(records,cache,conf,policy,matcher,output=None):\n    if cache.keys()!={r[\'id\'] for r in records}:raise ValueError(\'Cache não corresponde aos IDs da validação.\')\n    images=[];events=[]\n    for r in records:\n        im,ev=analyse(r,cache[r[\'id\']],conf,policy,matcher);images.append(im);events.extend(ev)\n    summary,classes=summarise(images,events,conf)\n    if output is not None:\n        output=Path(output);output.mkdir(parents=True,exist_ok=True)\n        core.save_json(output/\'metricas.json\',summary)\n        core.save_csv(output/\'metricas_por_classe.csv\',classes)\n        core.save_csv(output/\'alertas_por_imagem.csv\',images)\n        core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    return {\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n\ndef parse_current_events(events):\n    result=[]\n    for r in events:\n        v=dict(r)\n        for key in [\'bbox_ref\',\'bbox_pred\']:v[key]=json.loads(v[key]) if v[key] else None\n        result.append(v)\n    return result\n\ndef check_reproduction(result,original,cohort):\n    old={r[\'id\']:r for r in original[\'images\']};new={r[\'id\']:r for r in result[\'images\']}\n    if old.keys()!=new.keys():raise ValueError(\'Reprodução exige os mesmos IDs.\')\n    rows=[]\n    fields=[\'tp_alerta\',\'fn_alerta\',\'fp_alerta\',\'alta_ref\',\'alta_sem_alerta\',\'tp_detector\',\'fn_detector\',\'fp_detector\',\'gt_avaliaveis\',\'gt_ignorados\',\'predicoes_ignoradas_conf_025\']\n    for iid in sorted(old):\n        for key in fields:\n            key_new=\'predicoes_ignoradas_conf\' if key==\'predicoes_ignoradas_conf_025\' else key\n            a,b=old[iid][key],new[iid][key_new]\n            rows.append({\'coorte\':cohort,\'id\':iid,\'metrica\':key,\'notebook12\':a,\'notebook14\':b,\'igual\':a==b})\n    old_gt=data.gt_map(original[\'events\']);new_gt=data.gt_map(parse_current_events(result[\'events\']))\n    if old_gt.keys()!=new_gt.keys():raise ValueError(\'Referências GT mudaram na reprodução.\')\n    object_rows=[]\n    for key,a in old_gt.items():\n        b=new_gt[key]\n        fields_gt=[\'evento\',\'prioridade_ref\',\'prioridade_pred\',\'ha_alerta_ref\',\'ha_alerta_pred\']\n        diffs=[field for field in fields_gt if a[field]!=b[field]]\n        object_rows.append({\'coorte\':cohort,\'id\':key[0],\'classe\':a[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                            \'campos_diferentes\':\'|\'.join(diffs),\'igual\':not diffs})\n    return {\'coorte\':cohort,\'checagens_imagem\':len(rows),\'divergencias_imagem\':sum(not r[\'igual\'] for r in rows),\n            \'objetos_GT\':len(object_rows),\'divergencias_objetos\':sum(not r[\'igual\'] for r in object_rows),\n            \'reproduziu\':all(r[\'igual\'] for r in rows+object_rows)},rows,object_rows\n\ndef select_threshold(results,rule,reproduction_ok):\n    assessed=[]\n    for conf in THRESHOLDS:\n        home=results[\'home_val_historica\'][conf][\'summary\'];ext=results[\'openimages_val_nova\'][conf][\'summary\']\n        supported=all(s[\'alertas_por_objeto\'][k] is not None and math.isfinite(s[\'alertas_por_objeto\'][k])\n                      for s in [home,ext] for k in [\'precision\',\'recall\',\'f1\'])\n        if supported:\n            checks={**judge(home,ext,rule),\'metricas_com_suporte\':True}\n        else:\n            # Precisão sem nenhum alerta é indefinida; não vira zero ou aprovação.\n            checks={k:None for k in [\'home_precision_superior\',\'home_recall_preservado\',\'home_altas_preservadas\',\n                                    \'externa_precision_preservada\',\'externa_recall_preservado\',\'externa_altas_preservadas\']}\n            checks.update(elegivel=False,retencao_externa=False,metricas_com_suporte=False)\n        assessed.append({\'conf\':conf,\'home\':home,\'external\':ext,\'checks\':checks,\n                         \'comparavel_historico\':reproduction_ok,\'elegivel\':checks[\'elegivel\'] and reproduction_ok})\n    def ranking(r):\n        h=r[\'home\'][\'alertas_por_objeto\'];e=r[\'external\'][\'alertas_por_objeto\']\n        return (-h[\'precision\'],-h[\'recall\'],r[\'home\'][\'alta_sem_alerta\'],-e[\'f1\'],abs(r[\'conf\']-.25),-r[\'conf\'])\n    good=[r for r in assessed if r[\'elegivel\']]\n    chosen=min(good,key=ranking) if good else None\n    status=\'limiar_candidato_em_validacao\' if chosen else \'nenhum_limiar_elegivel\'\n    if not reproduction_ok:status=\'reproducao_divergente_sem_sugestao_de_limiar\'\n    return {\'status\':status,\'limiar_sugerido\':chosen[\'conf\'] if chosen else None,\n            \'comparabilidade_confirmada\':bool(reproduction_ok),\'assessed\':assessed,\n            \'limiar_galeria\':chosen[\'conf\'] if chosen else .10,\n            \'criterio_galeria_sem_elegivel\':\'0,10 fixado antes da inferência; somente diagnóstico.\',\n            \'modelo_em_uso_continua_baseline\':True,\'substituicao_automatica\':False}\n\ndef effects(results,bundle,cohort,cache):\n    old=data.gt_map(bundle[\'evaluations\'][\'candidato_10\'][cohort][\'events\'])\n    anchor=data.gt_map(parse_current_events(results[cohort][.25][\'events\']))\n    lost={k for k in anchor if old[k][\'ha_alerta_ref\'] and old[k][\'ha_alerta_pred\'] and not anchor[k][\'ha_alerta_pred\']}\n    changed=[];losses=[];priority=[]\n    for key in sorted(lost):\n        a,b=old[key],anchor[key]\n        if b[\'evento\']==\'correspondencia\':\n            priority.append({\'coorte\':cohort,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_candidato10\':a[\'prioridade_pred\'],\'prioridade_B_025\':b[\'prioridade_pred\'],\n                \'confianca_candidato10\':a[\'confianca\'],\'confianca_B_025\':b[\'confianca\'],\n                \'observacao\':\'Objeto ainda associado; alerta pode mudar pela geometria/prioridade.\'})\n    for conf in THRESHOLDS:\n        current=data.gt_map(parse_current_events(results[cohort][conf][\'events\']))\n        for key,b in current.items():\n            a=anchor[key]\n            if a[\'ha_alerta_ref\'] and a[\'ha_alerta_pred\']!=b[\'ha_alerta_pred\']:\n                changed.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                    \'alerta_B_025\':a[\'ha_alerta_pred\'],\'alerta_no_limiar\':b[\'ha_alerta_pred\'],\n                    \'prioridade_ref\':b[\'prioridade_ref\'],\'confianca_associada\':b[\'confianca\']})\n            if key not in lost:continue\n            same=[p for p in cache[key[0]] if p[\'classe_id\']==key[1]]\n            ious=core.iou_matrix([{\'bbox\':list(key[2])}],same)[0]\n            best=max(range(len(same)),key=lambda i:ious[i]) if same else None\n            losses.append({\'coorte\':cohort,\'conf\':conf,\'id\':key[0],\'classe\':b[\'classe\'],\'bbox_ref\':json.dumps(key[2]),\n                \'prioridade_ref\':b[\'prioridade_ref\'],\'recuperado\':b[\'ha_alerta_pred\'],\n                \'evento_B_025\':a[\'evento\'],\'evento_no_limiar\':b[\'evento\'],\n                \'melhor_iou_cache_mesma_classe\':float(ious[best]) if best is not None else None,\n                \'confianca_dessa_proposta\':same[best][\'confianca\'] if best is not None else None,\n                \'limite\':\'Melhor IoU não garante associação nem alerta; coluna recuperado usa avaliação completa.\'})\n    return changed,losses,priority\n\n\ndef validate_predictions(pred,record):\n    """Valida estrutura; preserva coordenadas numéricas como no avaliador 11/12.\n\n    Área zero e limites da imagem são diagnósticos, não filtros do cache.\n    Nenhuma proposta é removida, recortada ou corrigida aqui.\n    """\n    totals={\'predicoes\':len(pred),\'area_zero\':0,\'fora_da_imagem\':0,\n            \'area_zero_na_grade\':0,\'fora_da_imagem_na_grade\':0,\'amostras\':[]}\n    for index,p in enumerate(pred):\n        reason=None\n        if not isinstance(p,dict):\n            raise ValueError(f"Previsão não é um dicionário: {record[\'id\']}; índice={index}; valor={p!r}")\n        cid=p.get(\'classe_id\');box=p.get(\'bbox\');score=p.get(\'confianca\')\n        if type(cid)!=int or not 0<=cid<len(core.NAMES):reason=\'classe_id inválida\'\n        elif p.get(\'classe\')!=core.NAMES[cid]:reason=\'nome da classe incompatível\'\n        elif type(score) not in (int,float) or not math.isfinite(score) or not .001-1e-8<=score<=1:\n            reason=\'confiança não finita ou fora da faixa de captura\'\n        elif not isinstance(box,(list,tuple)) or len(box)!=4:reason=\'bbox deve ter quatro coordenadas\'\n        elif not all(type(v) in (int,float) and math.isfinite(v) for v in box):reason=\'coordenada não numérica ou não finita\'\n        elif box[2]<box[0] or box[3]<box[1]:reason=\'coordenadas xyxy invertidas\'\n        if reason:\n            raise ValueError(f"Previsão incompatível com o cache: {record[\'id\']}; motivo={reason}; "\n                f"índice={index}; classe_id={cid!r}; confiança={score!r}; bbox={box!r}; "\n                f"dimensões=(w={record[\'width\']}, h={record[\'height\']})")\n        zero=box[2]==box[0] or box[3]==box[1]\n        outside=(min(box)<-.01 or box[2]>record[\'width\']+.01 or box[3]>record[\'height\']+.01)\n        totals[\'area_zero\']+=int(zero);totals[\'fora_da_imagem\']+=int(outside)\n        totals[\'area_zero_na_grade\']+=int(zero and score>=min(THRESHOLDS))\n        totals[\'fora_da_imagem_na_grade\']+=int(outside and score>=min(THRESHOLDS))\n        if (zero or outside) and len(totals[\'amostras\'])<5:\n            totals[\'amostras\'].append({\'indice\':index,\'classe\':p[\'classe\'],\'confianca\':score,\n                \'bbox\':list(box),\'area_zero\':zero,\'fora_da_imagem\':outside,\n                \'width\':record[\'width\'],\'height\':record[\'height\']})\n    return totals\n\ndef save_geometry_diagnostics(directory,items):\n    totals={key:sum(item[\'diagnostico\'][key] for item in items) for key in\n            [\'predicoes\',\'area_zero\',\'fora_da_imagem\',\'area_zero_na_grade\',\'fora_da_imagem_na_grade\']}\n    unusual=[item for item in items if item[\'diagnostico\'][\'area_zero\'] or item[\'diagnostico\'][\'fora_da_imagem\']]\n    report={\'correcao\':\'14-cache-validacao-1.0.1\',\'imagens\':len(items),\n        \'imagens_com_geometria_incomum\':len(unusual),\'totais\':totals,\n        \'propostas_removidas\':0,\'coordenadas_alteradas\':False,\n        \'observacao\':\'Geometria incomum preservada no cache e no protocolo; não constitui objeto aprovado. Reprodução em 0,25 continua obrigatória.\',\n        \'casos\':unusual}\n    core.save_json(Path(directory)/\'diagnostico_geometria_previsoes.json\',report)\n    return {k:v for k,v in report.items() if k!=\'casos\'}\n\ndef obtain_cache(model,records,directory,weight_sha,environment,device,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    path=directory/\'predicoes_conf_0p001.jsonl.gz\';meta_path=directory/\'cache_manifesto.json\'\n    request=cache_request(records,weight_sha,environment)\n    if path.is_file() and meta_path.is_file():\n        meta=json.loads(meta_path.read_text())\n        if meta[\'solicitacao\']!=request or meta[\'cache_sha256\']!=core.sha256_file(path):\n            raise RuntimeError(\'Cache existente não corresponde aos pesos/entradas/ambiente. Não será reutilizado.\')\n        cached=read_cache(path,records)\n        diagnostics=[{\'id\':r[\'id\'],\'diagnostico\':validate_predictions(cached[r[\'id\']],r)} for r in records]\n        geometry=save_geometry_diagnostics(directory,diagnostics)\n        print(\'Cache conferido e reutilizado:\',records[0][\'coorte\'],flush=True)\n        return path,{**meta,\'geometria_resumo\':geometry,\'reutilizado_nesta_execucao\':True}\n    predict=predict_fn or core.model_predictions\n    tmp=path.with_name(path.name+\'.tmp\')\n    saturated=0;diagnostics=[]\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        for i,r in enumerate(records,1):\n            if r[\'role\'] not in {\'home_val\',\'val_nova\'}:raise ValueError(\'Papel proibido na inferência.\')\n            pred=predict(model,r,device)\n            try:diagnostic=validate_predictions(pred,r)\n            except ValueError as exc:\n                core.save_json(directory/\'previsao_rejeitada.json\',{\'id\':r[\'id\'],\'mensagem\':str(exc),\n                    \'width\':r[\'width\'],\'height\':r[\'height\'],\'propostas\':pred})\n                raise\n            diagnostics.append({\'id\':r[\'id\'],\'diagnostico\':diagnostic})\n            if len(pred)==PREDICT_ARGS[\'max_det\']:saturated+=1\n            f.write(json.dumps({\'id\':r[\'id\'],\'imagem\':r[\'name\'],\'image_sha256\':r[\'sha256\'],\n                                \'label_sha256\':r[\'label_sha256\'],\'predicoes\':pred},ensure_ascii=False)+\'\\n\')\n            if i%100==0 or i==len(records):print(f"Checkpoint / {r[\'coorte\']}: {i}/{len(records)}",flush=True)\n    tmp.replace(path)\n    geometry=save_geometry_diagnostics(directory,diagnostics)\n    meta={\'solicitacao\':request,\'cache_sha256\':core.sha256_file(path),\'concluido\':True,\n          \'validacao_previsoes\':\'14-cache-validacao-1.0.1\',\'geometria_resumo\':geometry,\n          \'imagens_atingiram_max_det\':saturated,\'limite\':\'Propostas pós-NMS em confiança 0,001; max_det=300, sem acesso aos logits brutos.\'}\n    core.save_json(meta_path,meta)\n    read_cache(path,records)\n    print(\'Diagnóstico geométrico do cache:\',geometry[\'totais\'],flush=True)\n    return path,{**meta,\'reutilizado_nesta_execucao\':False}\n', 'diag11_core.py': '"""Núcleo do diagnóstico 11. Não contém treinamento nem seleção de pesos."""\nfrom __future__ import annotations\n\nimport csv\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nfrom PIL import Image\n\nNAMES = ["bed", "sofa", "chair", "table", "lamp", "tv", "laptop",\n         "wardrobe", "window", "door", "potted plant", "photo frame"]\nALL_KNOWN = (1 << len(NAMES)) - 1\nALERT_IDS = {0, 1, 2, 3, 7, 10}\nALERT_MASK = sum(1 << c for c in ALERT_IDS)\nLEVEL = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef canonical(x):\n    return json.dumps(x, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(2**20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef json_clean(value):\n    if isinstance(value, dict):\n        return {str(k): json_clean(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [json_clean(v) for v in value]\n    if isinstance(value, np.generic):\n        return json_clean(value.item())\n    if isinstance(value, Path):\n        return str(value)\n    if isinstance(value, float) and not math.isfinite(value):\n        return None\n    return value\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(json_clean(value), ensure_ascii=False, indent=2,\n                              allow_nan=False), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef save_csv(path, rows, fields=None):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    rows = list(rows)\n    if fields is None:\n        fields = list(dict.fromkeys(k for row in rows for k in row))\n    with path.open("w", newline="", encoding="utf-8-sig") as f:\n        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="raise")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef safe_member(name):\n    p = PurePosixPath(name)\n    if p.is_absolute() or ".." in p.parts or "\\\\" in name:\n        raise ValueError(f"Membro inseguro no ZIP: {name}")\n    return p\n\n\ndef parse_yolo(raw, width, height, names=NAMES, source="label"):\n    objects = []\n    for line_no, line in enumerate(raw.decode("utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        parts = line.split()\n        if len(parts) != 5:\n            raise ValueError(f"{source}:{line_no}: esperados cinco campos.")\n        vals = [float(s) for s in parts]\n        if not all(math.isfinite(v) for v in vals):\n            raise ValueError(f"{source}:{line_no}: valor não finito.")\n        c, x, y, w, h = vals\n        if int(c) != c or not 0 <= c < len(names) or min(w, h) <= 0:\n            raise ValueError(f"{source}:{line_no}: classe ou dimensão inválida.")\n        box = [x-w/2, y-h/2, x+w/2, y+h/2]\n        if min(box) < -0.002 or max(box) > 1.002:\n            raise ValueError(f"{source}:{line_no}: caixa fora da imagem.")\n        box = [max(0, min(1, v)) for v in box]\n        objects.append({"classe_id": int(c), "classe": names[int(c)],\n                        "bbox": [box[0]*width, box[1]*height,\n                                 box[2]*width, box[3]*height],\n                        "ocluido": None, "truncado": None})\n    return objects\n\n\ndef hierarchy_mapping(cfg, class_rows, hierarchy):\n    if cfg["nomes"] != NAMES or len(cfg["mids_raizes"]) != len(NAMES):\n        raise ValueError("Classes da preparação incompatíveis com o projeto.")\n    descriptions = {r[0]: r[1] for r in class_rows if len(r) >= 2}\n    roots = cfg["mids_raizes"]\n    if any(mid not in descriptions for mid in roots):\n        raise ValueError("A descrição oficial não contém todas as raízes.")\n    descendants, ancestors = defaultdict(set), defaultdict(set)\n\n    def walk(node, parents=()):\n        mid = node["LabelName"]\n        ancestors[mid].update(parents)\n        descendants[mid].add(mid)\n        for parent in parents:\n            descendants[parent].add(mid)\n        for child in node.get("Subcategory", []):\n            walk(child, parents + (mid,))\n\n    walk(hierarchy)\n    excluded_names = set(cfg.get("ambiguas_excluidas", []))\n    excluded = {mid for mid, name in descriptions.items() if name in excluded_names}\n    candidates = defaultdict(set)\n    root_desc = {}\n    for cid, mid in enumerate(roots):\n        if mid not in descendants:\n            raise ValueError(f"Raiz ausente da hierarquia: {mid}")\n        root_desc[cid] = set(descendants[mid])\n        for sub in root_desc[cid]:\n            candidates[sub].add(cid)\n    ambiguous = excluded | {mid for mid, ids in candidates.items() if len(ids) > 1}\n    mapping = {mid: next(iter(ids)) for mid, ids in candidates.items()\n               if mid not in ambiguous}\n    return {"roots": roots, "descendants": root_desc, "ancestors": ancestors,\n            "mapping": mapping, "ambiguous": ambiguous,\n            "descriptions": descriptions,\n            "relevant_labels": set().union(*root_desc.values(),\n                                            *(ancestors[r] for r in roots))}\n\n\ndef prepare_oi_annotations(box_rows, label_rows, info):\n    """Cobertura conservadora: positivo na raiz ou negativo na raiz/ancestral.\n\n    Uma positiva apenas numa subclasse não prova cobertura exaustiva da classe\n    agregada. Grupos, representações, atributos desconhecidos ou caixas inválidas\n    tornam essa classe/imagem não avaliável, em vez de criarem falsos negativos.\n    """\n    labels = defaultdict(lambda: {"positive": set(), "negative": set()})\n    for row in label_rows:\n        if row.get("Source") not in {"verification", "crowdsource-verification"}:\n            continue\n        mid, confidence = row["LabelName"], float(row["Confidence"])\n        if mid not in info["relevant_labels"]:\n            continue\n        if confidence in (0, 1):\n            key = "positive" if confidence == 1 else "negative"\n            labels[row["ImageID"]][key].add(mid)\n    boxes, blocked = defaultdict(list), defaultdict(lambda: defaultdict(set))\n    audited = Counter()\n    for row in box_rows:\n        mid, iid = row["LabelName"], row["ImageID"]\n        affected = [cid for cid, mids in info["descendants"].items() if mid in mids]\n        if not affected:\n            continue\n        if mid in info["ambiguous"]:\n            for cid in affected:\n                blocked[iid][cid].add("subclasse_ambigua_excluida")\n            audited["caixas_subclasse_ambigua"] += 1\n            continue\n        cid = info["mapping"].get(mid)\n        if cid is None:\n            continue\n        flags = {k: int(float(row.get(k, -1))) for k in\n                 ["IsGroupOf", "IsDepiction", "IsOccluded", "IsTruncated"]}\n        reasons = []\n        if row.get("Source") not in {"xclick", "activemil"}:\n            reasons.append("origem_caixa_nao_reconhecida")\n        if flags["IsGroupOf"] != 0:\n            reasons.append("grupo_ou_atributo_desconhecido")\n        if flags["IsDepiction"] != 0:\n            reasons.append("representacao_ou_atributo_desconhecido")\n        xyxy = [float(row[k]) for k in ["XMin", "YMin", "XMax", "YMax"]]\n        if (not all(math.isfinite(v) for v in xyxy) or min(xyxy) < 0\n                or max(xyxy) > 1 or xyxy[2] <= xyxy[0] or xyxy[3] <= xyxy[1]):\n            reasons.append("coordenadas_invalidas")\n        if reasons:\n            blocked[iid][cid].update(reasons)\n            audited.update(reasons)\n            continue\n        boxes[iid].append({"classe_id": cid, "classe": NAMES[cid],\n                           "bbox_norm": xyxy, "mid_original": mid,\n                           "ocluido": flags["IsOccluded"] == 1 if flags["IsOccluded"] >= 0 else None,\n                           "truncado": flags["IsTruncated"] == 1 if flags["IsTruncated"] >= 0 else None})\n    records = {}\n    for iid in sorted(set(labels) | set(boxes)):\n        gt = []\n        seen = set()\n        for obj in boxes[iid]:\n            key = (obj["classe_id"], *(round(x, 8) for x in obj["bbox_norm"]))\n            if key not in seen:\n                seen.add(key)\n                gt.append(obj)\n        known, positive, negative, reasons = 0, 0, 0, {}\n        present = {o["classe_id"] for o in gt}\n        for cid, root in enumerate(info["roots"]):\n            pos = root in labels[iid]["positive"]\n            neg = bool(({root} | info["ancestors"][root]) & labels[iid]["negative"])\n            has_positive_label = bool(info["descendants"][cid] & labels[iid]["positive"])\n            reason = set(blocked[iid].get(cid, set()))\n            if neg and (cid in present or has_positive_label):\n                reason.add("contradicao_positivo_negativo")\n            if pos and cid not in present:\n                reason.add("positivo_sem_caixa_utilizavel")\n            if reason:\n                reasons[str(cid)] = sorted(reason)\n                continue\n            if neg:\n                known |= 1 << cid\n                negative |= 1 << cid\n            elif pos and cid in present:\n                known |= 1 << cid\n                positive |= 1 << cid\n        if positive or known & ALERT_MASK == ALERT_MASK:\n            records[iid] = {"id": iid, "source_split": "validation",\n                            "known_mask": known, "positive_mask": positive,\n                            "negative_mask": negative, "gt_normalizado": gt,\n                            "classes_positivas_avaliaveis": sorted(c for c in range(12)\n                                                                 if positive & (1 << c)),\n                            "restricoes_cobertura": reasons}\n    return records, dict(audited)\n\n\ndef iou_matrix(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)))\n    a = np.asarray([o["bbox"] for o in gt], dtype=float)\n    b = np.asarray([o["bbox"] for o in pred], dtype=float)\n    lt = np.maximum(a[:, None, :2], b[None, :, :2])\n    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])\n    intersection = np.maximum(rb-lt, 0).prod(axis=2)\n    aa = np.maximum(a[:, 2:]-a[:, :2], 0).prod(axis=1)\n    ba = np.maximum(b[:, 2:]-b[:, :2], 0).prod(axis=1)\n    return intersection / np.maximum(aa[:, None] + ba[None, :] - intersection, 1e-12)\n\n\ndef ap_matches(gt, pred, thresholds=np.linspace(.5, .95, 10)):\n    correct = np.zeros((len(pred), len(thresholds)), dtype=bool)\n    matrix = iou_matrix(gt, pred)\n    order = sorted(range(len(pred)), key=lambda pi: (-pred[pi]["confianca"], pi))\n    for ti, threshold in enumerate(thresholds):\n        used = set()\n        for pi in order:\n            options = [gi for gi, g in enumerate(gt) if gi not in used\n                       and g["classe_id"] == pred[pi]["classe_id"]\n                       and matrix[gi, pi] >= threshold]\n            if options:\n                gi = max(options, key=lambda k: (matrix[k, pi], -k))\n                used.add(gi)\n                correct[pi, ti] = True\n    return correct\n\n\ndef ratio(n, d):\n    return n/d if d else None\n\n\ndef alert_metrics(tp, fn, fp):\n    return {"tp": int(tp), "fn": int(fn), "fp": int(fp),\n            "precision": ratio(tp, tp+fp), "recall": ratio(tp, tp+fn),\n            "f1": ratio(2*tp, 2*tp+fp+fn)}\n\n\ndef alert_on(obj):\n    return bool(obj and LEVEL[obj["prioridade"]] >= LEVEL["medio"])\n\n\ndef with_policy(objects, width, height, policy):\n    return [{**o, **policy.avaliar_bbox(o["classe"], o["bbox"], width, height,\n                                       versao="v2")} for o in objects]\n\n\ndef record_event(record, model_name, event, g=None, p=None, iou=None, all_pred=()):\n    gbox = g["bbox"] if g else None\n    pbox = p["bbox"] if p else None\n    area = ((gbox[2]-gbox[0])*(gbox[3]-gbox[1]) /\n            (record["width"]*record["height"])) if gbox else None\n    side_at_640 = (min(gbox[2]-gbox[0], gbox[3]-gbox[1]) *\n                   640/max(record["width"], record["height"])) if gbox else None\n    flags = []\n    near_best = None\n    if event == "objeto_nao_detectado":\n        if side_at_640 < 32:\n            flags.append("objeto_pequeno_na_entrada_640")\n        if g.get("ocluido") is True:\n            flags.append("oclusao_anotada")\n        if g.get("truncado") is True:\n            flags.append("truncamento_anotado")\n        overlaps = iou_matrix([g], all_pred)[0] if all_pred else []\n        same = [i for i, o in enumerate(all_pred) if o["classe_id"] == g["classe_id"]]\n        if same:\n            best = max(same, key=lambda i: overlaps[i])\n            near_best = float(overlaps[best])\n            if overlaps[best] >= .5 and all_pred[best]["confianca"] < .25:\n                flags.append("deteccao_abaixo_da_confianca_025")\n            elif .1 <= overlaps[best] < .5:\n                flags.append("caixa_mesma_classe_iou_010_a_050")\n        if any(overlaps[i] >= .5 and o["classe_id"] != g["classe_id"]\n               and o["confianca"] >= .25 for i, o in enumerate(all_pred)):\n            flags.append("sobreposicao_com_previsao_de_outra_classe")\n        if not flags:\n            flags.append("omissao_sem_indicio_automatico")\n    if event == "previsao_nao_associada":\n        flags.append("previsao_sem_correspondencia_na_classe_avaliavel")\n    return {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n            "modelo": model_name, "evento": event,\n            "classe_id": g["classe_id"] if g else p["classe_id"],\n            "classe": g["classe"] if g else p["classe"],\n            "bbox_ref": json.dumps(gbox) if g else "",\n            "bbox_pred": json.dumps(pbox) if p else "",\n            "iou": iou, "confianca": p["confianca"] if p else None,\n            "prioridade_ref": g["prioridade"] if g else "sem_prioridade",\n            "prioridade_pred": p["prioridade"] if p else "sem_prioridade",\n            "ha_alerta_ref": alert_on(g), "ha_alerta_pred": alert_on(p),\n            "area_relativa_gt": area, "menor_lado_gt_em_640": side_at_640,\n            "oclusao_anotada": g.get("ocluido") if g else None,\n            "truncamento_anotado": g.get("truncado") if g else None,\n            "melhor_iou_mesma_classe": near_best,\n            "indicios": "|".join(flags)}\n\n\ndef analyse_prediction(record, pred_low, model_name, policy, matcher):\n    if record.get("role") not in {"home_val", "val_nova"}:\n        raise ValueError("Somente validação pode entrar na inferência do notebook 11.")\n    known = int(record["known_mask"])\n    gt = with_policy([g for g in record["gt"] if known & (1 << g["classe_id"])],\n                     record["width"], record["height"], policy)\n    pred_low_known = [p for p in pred_low if known & (1 << p["classe_id"])]\n    pred_all = with_policy([p for p in pred_low if p["confianca"] >= .25],\n                           record["width"], record["height"], policy)\n    pred = [p for p in pred_all if known & (1 << p["classe_id"])]\n    pairs, missing, extras = matcher(gt, pred)\n    events = []\n    for gi, pi, iou in pairs:\n        events.append(record_event(record, model_name, "correspondencia",\n                                   gt[gi], pred[pi], float(iou)))\n    for gi in missing:\n        events.append(record_event(record, model_name, "objeto_nao_detectado",\n                                   g=gt[gi], all_pred=pred_low_known))\n    for pi in extras:\n        events.append(record_event(record, model_name, "previsao_nao_associada", p=pred[pi]))\n    tp = sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    fn = sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in events)\n    fp = sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in events)\n    high_n = sum(g["prioridade"] == "alto" for g in gt)\n    high_missing = sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in events)\n    messages = [policy.gerar_mensagem(p, "medio") for p in pred_all if alert_on(p)]\n    if any(not isinstance(m, str) or not m.strip() for m in messages):\n        raise RuntimeError("A política produziu alerta sem texto.")\n    image = {"coorte": record["coorte"], "imagem": record["name"], "id": record["id"],\n             "modelo": model_name, "tp_alerta": tp, "fn_alerta": fn, "fp_alerta": fp,\n             "alta_ref": high_n, "alta_sem_alerta": high_missing,\n             "tp_detector": len(pairs), "fn_detector": len(missing), "fp_detector": len(extras),\n             "gt_avaliaveis": len(gt), "gt_ignorados": len(record["gt"])-len(gt),\n             "predicoes_ignoradas_conf_025": len(pred_all)-len(pred),\n             "cobertura_completa_alertas": known & ALERT_MASK == ALERT_MASK,\n             "ha_alerta_ref": any(alert_on(g) for g in gt),\n             "ha_alerta_pred": any(alert_on(p) for p in pred),\n             "mensagem_alerta": "\\n".join(messages)}\n    stats = (ap_matches(gt, pred_low_known),\n             np.array([p["confianca"] for p in pred_low_known], dtype=float),\n             np.array([p["classe_id"] for p in pred_low_known], dtype=int),\n             np.array([g["classe_id"] for g in gt], dtype=int))\n    return image, events, stats\n\n\ndef summarise(images, events, stats, ap_function=None):\n    if not images:\n        raise ValueError("Coorte de avaliação vazia.")\n    total = lambda k: sum(int(r[k]) for r in images)\n    output = {"imagens": len(images), "conf_alertas": .25,\n              "alertas_por_objeto": alert_metrics(total("tp_alerta"), total("fn_alerta"), total("fp_alerta")),\n              "alta_ref": total("alta_ref"), "alta_sem_alerta": total("alta_sem_alerta"),\n              "taxa_alta_sem_alerta": ratio(total("alta_sem_alerta"), total("alta_ref")),\n              "detector_conf_025": alert_metrics(total("tp_detector"), total("fn_detector"), total("fp_detector")),\n              "gt_ignorados_sem_cobertura": total("gt_ignorados"),\n              "predicoes_ignoradas_conf_025": total("predicoes_ignoradas_conf_025")}\n    complete = [r for r in images if r["cobertura_completa_alertas"]]\n    output["imagens_com_cobertura_completa_alertas"] = len(complete)\n    output["alertas_por_imagem_cobertura_completa"] = alert_metrics(\n        sum(r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete),\n        sum(r["ha_alerta_ref"] and not r["ha_alerta_pred"] for r in complete),\n        sum(not r["ha_alerta_ref"] and r["ha_alerta_pred"] for r in complete))\n    correct = np.concatenate([s[0] for s in stats], axis=0)\n    scores = np.concatenate([s[1] for s in stats])\n    pred_cls = np.concatenate([s[2] for s in stats])\n    target_cls = np.concatenate([s[3] for s in stats])\n    ap = {}\n    if ap_function is not None and len(target_cls) and len(pred_cls):\n        result = ap_function(correct, scores, pred_cls, target_cls, plot=False,\n                             names=dict(enumerate(NAMES)))\n        values, ids = np.asarray(result[5]), np.asarray(result[6], dtype=int)\n        if values.shape != (len(ids), 10):\n            raise RuntimeError("Retorno de AP incompatível com a versão fixada.")\n        ap = {int(c): a for c, a in zip(ids, values)}\n    elif ap_function is not None and len(target_cls):\n        ap = {int(c): np.zeros(10) for c in np.unique(target_cls)}\n    output["map50_recorte"] = float(np.mean([a[0] for a in ap.values()])) if ap else None\n    output["map50_95_recorte"] = float(np.mean(list(ap.values()))) if ap else None\n    class_rows = []\n    for cid, name in enumerate(NAMES):\n        part = [e for e in events if e["classe_id"] == cid]\n        a = ap.get(cid)\n        n_gt = int((target_cls == cid).sum())\n        class_rows.append({"classe_id": cid, "classe": name, "gt_avaliaveis": n_gt,\n                           "ap50": float(a[0]) if a is not None else None,\n                           "ap50_95": float(a.mean()) if a is not None else None,\n                           "suporte_reduzido": n_gt < 30,\n                           **alert_metrics(sum(e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part),\n                                           sum(e["ha_alerta_ref"] and not e["ha_alerta_pred"] for e in part),\n                                           sum(not e["ha_alerta_ref"] and e["ha_alerta_pred"] for e in part)),\n                           "alta_ref": sum(e["prioridade_ref"] == "alto" for e in part),\n                           "alta_sem_alerta": sum(e["prioridade_ref"] == "alto" and not e["ha_alerta_pred"] for e in part)})\n    return output, class_rows\n\n\ndef model_predictions(model, record, device):\n    if record["role"] not in {"home_val", "val_nova"}:\n        raise ValueError("Tentativa de inferência em reserva ou conjunto não autorizado.")\n    result = model.predict(str(record["image"]), conf=.001, imgsz=640, rect=True,\n                           iou=.70, max_det=300, device=device, half=False,\n                           verbose=False, save=False)[0]\n    if Path(result.path).resolve() != Path(record["image"]).resolve():\n        raise RuntimeError("A inferência retornou uma imagem diferente.")\n    if result.boxes is None:\n        return []\n    return [{"classe_id": int(c), "classe": NAMES[int(c)],\n             "bbox": [float(v) for v in box], "confianca": float(score)}\n            for c, box, score in zip(result.boxes.cls.cpu().numpy(),\n                                      result.boxes.xyxy.cpu().numpy(),\n                                      result.boxes.conf.cpu().numpy())]\n\n\ndef evaluate_model(model, records, model_name, output_dir, policy, matcher, device,\n                   ap_function):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    images, events, stats = [], [], []\n    cache_file = output_dir / "predicoes_conf_0001.jsonl.gz"\n    with gzip.open(cache_file, "wt", encoding="utf-8") as cache:\n        for i, record in enumerate(records, 1):\n            pred = model_predictions(model, record, device)\n            cache.write(json.dumps({"id": record["id"], "imagem": record["name"],\n                                    "predicoes": pred}, ensure_ascii=False) + "\\n")\n            im, ev, st = analyse_prediction(record, pred, model_name, policy, matcher)\n            images.append(im)\n            events.extend(ev)\n            stats.append(st)\n            if i % 100 == 0 or i == len(records):\n                print(f"{record[\'coorte\']} / {model_name}: {i}/{len(records)}", flush=True)\n    summary, classes = summarise(images, events, stats, ap_function)\n    summary.update(modelo=model_name, coorte=records[0]["coorte"],\n                   protocolo="12 classes; AP por confiança; alertas por IoU; cobertura conservadora",\n                   nao_e_openimages_challenge=True, selecao_automatica_de_modelo=False)\n    save_json(output_dir / "metricas.json", summary)\n    save_csv(output_dir / "metricas_por_classe.csv", classes)\n    save_csv(output_dir / "alertas_por_imagem.csv", images)\n    save_csv(output_dir / "avaliacao_objetos.csv", events)\n    (output_dir / "alertas_texto.txt").write_text(\n        "\\n\\n".join(r["imagem"]+"\\n"+r["mensagem_alerta"] for r in images if r["mensagem_alerta"]),\n        encoding="utf-8")\n    return {"summary": summary, "classes": classes, "images": images, "events": events,\n            "cache_file": cache_file}\n', 'exp12_protocol.py': '"""Orçamento e seleção do experimento 12, sem dependência de Torch."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nimport hashlib\nimport math\nimport numpy as np\n\nARMS = {\n    "A_home": {"home_per_batch": 16, "oi_per_batch": 0},\n    "B_home_replay": {"home_per_batch": 12, "oi_per_batch": 4},\n}\n\n\nclass BudgetSampler:\n    """Fluxos por fonte independentes; quantidade fixa de batches por época."""\n    def __init__(self, home, oi, batch_size, batches, oi_per_batch, seed=42):\n        self.home, self.oi = list(home), list(oi)\n        self.batch_size, self.batches = int(batch_size), int(batches)\n        self.oi_per_batch = int(oi_per_batch)\n        self.home_per_batch = self.batch_size - self.oi_per_batch\n        self.seed, self.epoch = int(seed), 0\n        if (not self.home or self.batches <= 0 or self.batch_size <= 0 or\n                not 0 <= self.oi_per_batch < self.batch_size or\n                (self.oi_per_batch and not self.oi) or set(self.home) & set(self.oi)):\n            raise ValueError("Fontes ou orçamento inválidos.")\n\n    def __len__(self):\n        return self.batches*self.batch_size\n\n    def set_epoch(self, epoch):\n        self.epoch = int(epoch)\n\n    def _stream(self, values, needed, source):\n        if not needed:\n            return []\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, source]))\n        result = []\n        while len(result) < needed:\n            result.extend(rng.permutation(values).tolist())\n        return result[:needed]\n\n    def __iter__(self):\n        home = self._stream(self.home, self.batches*self.home_per_batch, 0)\n        oi = self._stream(self.oi, self.batches*self.oi_per_batch, 1)\n        rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, 2]))\n        for b in range(self.batches):\n            batch = home[b*self.home_per_batch:(b+1)*self.home_per_batch]\n            batch += oi[b*self.oi_per_batch:(b+1)*self.oi_per_batch]\n            rng.shuffle(batch)\n            yield from batch\n\n\ndef order_id(iid, seed=42):\n    return hashlib.sha256(f"{seed}|{iid}".encode()).hexdigest(), iid\n\n\ndef diverse_subset(rows, n=2000, seed=42, per_class=40):\n    if n <= 0 or n > len(rows) or len({r["id"] for r in rows}) != len(rows):\n        raise ValueError("Tamanho do replay ou identidade dos IDs inválidos.")\n    mapping = {r["id"]: r for r in rows}\n    groups = {c: sorted([r["id"] for r in rows if int(r["classe_primaria"]) == c],\n                        key=lambda iid: order_id(iid, seed)) for c in range(12)}\n    chosen = set()\n    for k in range(per_class):\n        for c in sorted(groups, key=lambda cid: (len(groups[cid]), cid)):\n            if k < len(groups[c]) and len(chosen) < n:\n                chosen.add(groups[c][k])\n    for iid in sorted(mapping, key=lambda x: order_id(x, seed)):\n        if len(chosen) == n:\n            break\n        chosen.add(iid)\n    return [mapping[iid] for iid in sorted(chosen)]\n\n\ndef choose_replay(history_rows, n=2200, seed=42, per_class=40, forbidden_hashes=()):\n    batches = {f"lote_{i:06}.zip" for i in range(1, 6)}\n    rows = [r for r in history_rows if r["lote"] in batches]\n    if len(rows) != 5000 or any(r["split"] != "train" for r in rows):\n        raise ValueError("Os cinco lotes do piloto precisam conter 5.000 imagens só de treino.")\n    forbidden = set(forbidden_hashes)\n    return diverse_subset([r for r in rows if r["imagem_sha256"] not in forbidden],\n                          n, seed, per_class)\n\n\ndef validate_metric_block(block, label):\n    for k in ["precision", "recall", "f1"]:\n        x = block["alertas_por_objeto"][k]\n        if x is None or not math.isfinite(x):\n            raise ValueError(f"Métrica sem suporte ou não finita: {label}/{k}")\n    for k in ["alta_ref", "alta_sem_alerta"]:\n        if int(block[k]) < 0:\n            raise ValueError("Contagem inválida.")\n\n\ndef make_rule(baseline_home, candidate_external, dataset_sha, policy_sha):\n    validate_metric_block(baseline_home, "baseline_home")\n    validate_metric_block(candidate_external, "candidate_external")\n    return {\n        "versao": "12-regra-duas-validacoes-v1",\n        "home": {\n            "precision_maior_que": baseline_home["alertas_por_objeto"]["precision"],\n            "recall_minimo": baseline_home["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": baseline_home["alta_sem_alerta"],\n            "objetos_alerta_ref": baseline_home["alertas_por_objeto"]["tp"]+baseline_home["alertas_por_objeto"]["fn"],\n            "alta_ref": baseline_home["alta_ref"],\n        },\n        "externa": {\n            "precision_minima": candidate_external["alertas_por_objeto"]["precision"],\n            "recall_minimo": candidate_external["alertas_por_objeto"]["recall"],\n            "alta_sem_alerta_maximo": candidate_external["alta_sem_alerta"],\n            "objetos_alerta_ref": candidate_external["alertas_por_objeto"]["tp"]+candidate_external["alertas_por_objeto"]["fn"],\n            "alta_ref": candidate_external["alta_ref"],\n        },\n        "dataset_sha256": dataset_sha, "politica_sha256": policy_sha,\n        "conf_alertas": .25, "testes_consultados": False,\n        "tolerancia_numerica": 1e-12,\n        "criterio": "Home supera precisão do baseline, preserva recall e omissões altas; externa preserva precisão, recall e omissões altas do candidato 10.",\n        "desempate": "Precisão Home, recall Home, omissões altas Home, F1 externa, menos atualizações, identificador.",\n        "sem_elegivel": "Preservar baseline em uso; candidato 10 e todos os checkpoints continuam disponíveis.",\n    }\n\n\ndef judge(home, external, rule):\n    validate_metric_block(home, "home")\n    validate_metric_block(external, "external")\n    h, e = home["alertas_por_objeto"], external["alertas_por_objeto"]\n    rh, re = rule["home"], rule["externa"]\n    for observed, expected in [(h["tp"]+h["fn"], rh["objetos_alerta_ref"]),\n                               (e["tp"]+e["fn"], re["objetos_alerta_ref"]),\n                               (home["alta_ref"], rh["alta_ref"]),\n                               (external["alta_ref"], re["alta_ref"])]:\n        if observed != expected:\n            raise ValueError("Os denominadores de referência mudaram entre checkpoints.")\n    eps = rule["tolerancia_numerica"]\n    checks = {\n        "home_precision_superior": h["precision"] > rh["precision_maior_que"]+eps,\n        "home_recall_preservado": h["recall"] >= rh["recall_minimo"]-eps,\n        "home_altas_preservadas": home["alta_sem_alerta"] <= rh["alta_sem_alerta_maximo"],\n        "externa_precision_preservada": e["precision"] >= re["precision_minima"]-eps,\n        "externa_recall_preservado": e["recall"] >= re["recall_minimo"]-eps,\n        "externa_altas_preservadas": external["alta_sem_alerta"] <= re["alta_sem_alerta_maximo"],\n    }\n    return {**checks, "elegivel": all(checks.values()),\n            "retencao_externa": all(v for k, v in checks.items() if k.startswith("externa_"))}\n\n\ndef select_checkpoints(entries, rule):\n    if not entries:\n        raise ValueError("Nenhum checkpoint avaliado.")\n    assessed = []\n    for entry in entries:\n        checks = judge(entry["home"], entry["external"], rule)\n        assessed.append({**entry, "checks": checks})\n    eligible = [r for r in assessed if r["checks"]["elegivel"]]\n\n    def ranking(r):\n        h = r["home"]["alertas_por_objeto"]\n        return (-h["precision"], -h["recall"], r["home"]["alta_sem_alerta"],\n                -r["external"]["alertas_por_objeto"]["f1"], r["updates"], r["id"])\n\n    selected = min(eligible, key=ranking) if eligible else None\n    # Apenas para inspeção. A classificação exploratória não muda os requisitos.\n    retained = [r for r in assessed if r["checks"]["retencao_externa"]]\n    exploratory = min(retained, key=lambda r: (\n        -r["home"]["alertas_por_objeto"]["recall"],\n        r["home"]["alta_sem_alerta"],\n        -r["home"]["alertas_por_objeto"]["precision"],\n        -r["external"]["alertas_por_objeto"]["f1"],\n        r["updates"], r["id"])) if retained else None\n    return {"eligible": selected, "exploratory": exploratory, "assessed": assessed}\n', 'final20_data.py': '"""Reserva final: somente os 300 IDs e as anotações do manifesto autenticado."""\nfrom __future__ import annotations\nfrom collections import Counter\nimport gzip,hashlib,json,math,zipfile\nfrom pathlib import Path\nfrom PIL import Image\nimport diag11_core as core\n\nMODELS=(\'baseline\',\'candidato_10\')\nROLE=\'reserva_futura\'\nCOHORT=\'openimages_reserva_final\'\n\ndef digest(value):\n    return hashlib.sha256(core.canonical(value).encode()).hexdigest()\n\ndef save_gz(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    with gzip.open(tmp,\'wt\',encoding=\'utf-8\',compresslevel=4) as f:\n        json.dump(core.json_clean(value),f,ensure_ascii=False,allow_nan=False)\n    tmp.replace(path)\n\ndef source19(path,expected,frozen,output):\n    if core.sha256_file(path)!=expected[\'pacote_19\']:raise ValueError(\'O pacote 19 difere do resultado analisado.\')\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(path) as z:\n        if len(set(z.namelist()))!=len(z.namelist()) or z.testzip():raise ValueError(\'Pacote 19 inválido.\')\n        for name in z.namelist():core.safe_member(name)\n        summary=json.loads(z.read(\'resumo_consolidacao_19.json\'))\n        plan=json.loads(z.read(\'protocolo_avaliacao_final.json\'))\n        reg=json.loads(z.read(\'registro_protocolo_final.json\'))\n        if (not summary[\'concluido\'] or summary[\'context_key\']!=expected[\'contexto_19\']\n            or plan!=frozen or digest(plan)!=expected[\'protocolo_sha256\']\n            or reg[\'sha256_canonico\']!=digest(plan) or reg[\'reserva_consultada\']\n            or summary[\'reserva_futura_extraida\'] or summary[\'reserva_futura_inferida\']):\n            raise ValueError(\'O protocolo congelado no 19 não corresponde à avaliação planejada.\')\n        development=json.loads(gzip.decompress(z.read(\'dados_validacao_reutilizados.json.gz\')))\n        if Counter(r[\'role\'] for r in development)!={\'home_val\':456,\'val_nova\':1000}:\n            raise ValueError(\'Referências de desenvolvimento incompatíveis.\')\n        for name in [\'protocolo_avaliacao_final.json\',\'registro_protocolo_final.json\',\n                     \'comparacao_finais_e_referencias.csv\',\'resumo_hipoteses_16_18.csv\']:\n            target=output/\'fontes19\'/name;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))\n    return plan,development\n\ndef extract_reserve(archive,plan,development,destination,output):\n    if core.sha256_file(archive)!=plan[\'archive_sha256\']:raise ValueError(\'ZIP da reserva diferente do preservado.\')\n    destination=Path(destination);output=Path(output)\n    records=[];receipts=[]\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist()\n        if len(names)!=len(set(names)):raise ValueError(\'ZIP com nomes repetidos.\')\n        for name in names:core.safe_member(name)\n        meta=json.loads(z.read(\'manifesto_validacao_nova.json\'))\n        claimed=meta[\'dataset_sha256\']\n        if claimed!=plan[\'dataset_sha256\'] or digest({k:v for k,v in meta.items() if k!=\'dataset_sha256\'})!=claimed:\n            raise ValueError(\'Manifesto da reserva foi alterado.\')\n        items=meta[\'arquivos\'];counts=Counter(r[\'role\'] for r in items)\n        if (meta[\'nomes\']!=core.NAMES or counts!={\'val_nova\':1000,ROLE:plan[\'images\']}\n            or meta[\'contagens\']!=dict(counts) or len({r[\'id\'] for r in items})!=len(items)):\n            raise ValueError(\'Mapeamento, divisão ou IDs incompatíveis com o plano congelado.\')\n        dev={r[\'id\']:r for r in development if r[\'role\']==\'val_nova\'}\n        frozen_val={r[\'id\']:r for r in items if r[\'role\']==\'val_nova\'}\n        if set(dev)!=set(frozen_val):raise ValueError(\'A validação anterior não pertence a esta divisão.\')\n        for iid,r in frozen_val.items():\n            for field in [\'sha256\',\'label_sha256\',\'known_mask\',\'width\',\'height\']:\n                if r[field]!=dev[iid][field]:raise ValueError(\'A divisão anterior mudou: \'+iid+\'/\'+field)\n        selected=sorted((r for r in items if r[\'role\']==ROLE),key=lambda r:r[\'id\'])\n        if len(selected)!=plan[\'images\']:raise ValueError(\'A reserva não tem todos os 300 IDs previstos.\')\n        if {r[\'id\'] for r in selected}&set(dev):raise ValueError(\'IDs da reserva apareceram na validação.\')\n        dev_sha={r[\'sha256\'] for r in development}\n        if any(r[\'sha256\'] in dev_sha for r in selected):raise ValueError(\'Bytes da reserva já aparecem no desenvolvimento.\')\n        for index,item in enumerate(selected,1):\n            iid=item[\'id\']\n            if (not isinstance(iid,str) or len(iid)!=16 or any(c not in \'0123456789abcdef\' for c in iid)):\n                raise ValueError(\'ID externo inválido.\')\n            expected_members={\'image\':f\'images/{ROLE}/oi_val_{iid}.jpg\',\n                              \'label\':f\'labels/{ROLE}/oi_val_{iid}.txt\',\n                              \'coverage\':f\'coverage/{ROLE}/oi_val_{iid}.json\'}\n            extracted={}\n            for field,member in expected_members.items():\n                if item[field]!=member:raise ValueError(\'Arquivo fora da reserva: \'+item[field])\n                raw=z.read(member);key=\'sha256\' if field==\'image\' else field+\'_sha256\'\n                if hashlib.sha256(raw).hexdigest()!=item[key]:raise ValueError(\'Bytes alterados: \'+member)\n                target=destination/member;target.parent.mkdir(parents=True,exist_ok=True)\n                if not target.is_file() or core.sha256_file(target)!=item[key]:target.write_bytes(raw)\n                extracted[field]=target\n                receipts.append({\'id\':iid,\'papel\':ROLE,\'tipo\':field,\'member\':member,\'sha256\':item[key]})\n            with Image.open(extracted[\'image\']) as im:\n                if im.size!=(item[\'width\'],item[\'height\']):raise ValueError(\'Dimensões diferentes do manifesto.\')\n                im.verify()\n            coverage=json.loads(extracted[\'coverage\'].read_text())\n            for key in [\'known_mask\',\'positive_mask\',\'negative_mask\']:\n                if (type(item[key])!=int or not 0<=item[key]<=core.ALL_KNOWN or coverage[key]!=item[key]):\n                    raise ValueError(\'Cobertura de referência divergente: \'+iid)\n            if item[\'positive_mask\']&item[\'negative_mask\'] or (item[\'positive_mask\']|item[\'negative_mask\'])!=item[\'known_mask\']:\n                raise ValueError(\'Máscaras de cobertura contraditórias.\')\n            ph=item[\'phash\']\n            if not isinstance(ph,str) or len(ph)!=16 or any(c not in \'0123456789abcdef\' for c in ph):\n                raise ValueError(\'pHash64 congelado inválido.\')\n            gt=[]\n            for g in item[\'gt_normalizado\']:\n                cid=g[\'classe_id\'];box=g[\'bbox_norm\']\n                if (type(cid)!=int or not 0<=cid<len(core.NAMES) or g[\'classe\']!=core.NAMES[cid]\n                    or len(box)!=4 or not all(math.isfinite(v) for v in box)\n                    or min(box)<0 or max(box)>1 or box[2]<=box[0] or box[3]<=box[1]):\n                    raise ValueError(\'Caixa de referência inválida: \'+iid)\n                gt.append({**g,\'bbox\':[box[0]*item[\'width\'],box[1]*item[\'height\'],box[2]*item[\'width\'],box[3]*item[\'height\']]})\n            parsed=core.parse_yolo(extracted[\'label\'].read_bytes(),item[\'width\'],item[\'height\'])\n            if len(parsed)!=len(gt) or any(a[\'classe_id\']!=b[\'classe_id\'] or max(abs(x-y) for x,y in zip(a[\'bbox\'],b[\'bbox\']))>1e-5 for a,b in zip(parsed,gt)):\n                raise ValueError(\'Anotações YOLO divergem das caixas congeladas: \'+iid)\n            records.append({\'id\':iid,\'name\':extracted[\'image\'].name,\'image\':extracted[\'image\'],\n                \'label\':extracted[\'label\'],\'coverage\':extracted[\'coverage\'],\'role\':ROLE,\'coorte\':COHORT,\n                **{k:item[k] for k in [\'width\',\'height\',\'known_mask\',\'positive_mask\',\'negative_mask\',\'sha256\',\'label_sha256\',\'coverage_sha256\']},\n                \'phash\':int(ph,16),\'phash_origem\':\'manifesto autenticado do notebook 11\',\'gt\':gt,\'creditos\':item.get(\'creditos\',{})})\n            if index%100==0:print(f\'Reserva conferida: {index}/{len(selected)}\',flush=True)\n    near=[]\n    for r in records:\n        found=next((d for d in development if (r[\'phash\']^int(d[\'phash\'])).bit_count()<=4),None)\n        if found:near.append({\'id_reserva\':r[\'id\'],\'id_desenvolvimento\':found[\'id\'],\'distancia\':(r[\'phash\']^int(found[\'phash\'])).bit_count()})\n    audit={\'imagens\':len(records),\'dataset_sha256\':plan[\'dataset_sha256\'],\'archive_sha256\':plan[\'archive_sha256\'],\n        \'ids_substituidos\':0,\'imagens_excluidas\':0,\'intersecao_ids_val_anterior\':0,\'intersecao_bytes_desenvolvimento\':0,\n        \'pares_semelhanca_phash_com_desenvolvimento\':near,\'semelhanca_phash_nao_prova_duplicacao\':True,\n        \'train_extraido\':False,\'home_test_extraido\':False,\'oi_test20k_extraido\':False,\n        \'origem_treino_original_baseline\':\'nao_determinado\',\'recibos\':receipts}\n    core.save_json(output/\'extracao_reserva_20.json\',audit)\n    save_gz(output/\'registros_reserva_20.json.gz\',[{k:v for k,v in r.items() if k not in {\'image\',\'label\',\'coverage\'}} for r in records])\n    return records,audit\n', 'final20_drive.py': '"""Drive, recibos atômicos e pacote leve; credenciais ficam fora dos resultados."""\nfrom __future__ import annotations\nimport base64\nimport configparser\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport zipfile\n\nimport diag11_core as core\n\nclass DigestMismatch(RuntimeError):\n    pass\n\nclass Drive:\n    def __init__(self, binary, config, root=\'gdrive_tcc:TCC_Visao_Computacional\'):\n        self.binary, self.config, self.root = str(binary), str(config), root\n\n    @classmethod\n    def from_kaggle(cls):\n        from kaggle_secrets import UserSecretsClient\n        root = Path(\'/tmp/tcc_rclone\')\n        root.mkdir(parents=True, exist_ok=True, mode=0o700)\n        root.chmod(0o700)\n        try:\n            secret = UserSecretsClient().get_secret(\'GDRIVE_RCLONE_CONFIG_B64\')\n        except Exception:\n            raise RuntimeError(\'Habilite o Secret existente GDRIVE_RCLONE_CONFIG_B64 em Add-ons → Secrets.\') from None\n        try:\n            raw = base64.b64decode(secret.strip(), validate=True)\n            cfg = configparser.ConfigParser(interpolation=None)\n            cfg.read_string(raw.decode(\'utf-8-sig\'))\n        except Exception:\n            raise RuntimeError(\'A configuração do Secret rclone não é válida.\') from None\n        if (not cfg.has_section(\'gdrive_tcc\') or cfg.get(\'gdrive_tcc\', \'type\', fallback=\'\') != \'drive\'\n                or not cfg.get(\'gdrive_tcc\', \'client_id\', fallback=\'\')\n                or not cfg.get(\'gdrive_tcc\', \'token\', fallback=\'\')\n                or cfg.get(\'gdrive_tcc\', \'scope\', fallback=\'\') != \'drive\'):\n            raise RuntimeError(\'O Secret precisa da conexão gdrive_tcc autorizada para leitura e gravação em Meu Drive.\')\n        cfg.remove_option(\'gdrive_tcc\', \'team_drive\')\n        config = root / \'gdrive_tcc.conf\'\n        with config.open(\'w\', encoding=\'utf-8\') as f:\n            cfg.write(f)\n        config.chmod(0o600)\n        del secret, raw, cfg\n        binary = root / \'rclone\'\n        if not binary.is_file():\n            executable = shutil.which(\'rclone\')\n            if not executable:\n                import os\n                prefix = [] if os.geteuid() == 0 else [\'sudo\']\n                subprocess.run(prefix + [\'apt-get\', \'update\', \'-qq\'], check=True)\n                subprocess.run(prefix + [\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'rclone\'], check=True)\n                executable = shutil.which(\'rclone\')\n            if not executable:\n                raise RuntimeError(\'Executável rclone indisponível após a instalação.\')\n            shutil.copy2(executable, binary)\n            binary.chmod(0o700)\n        return cls(binary, config)\n\n    def _call(self, args):\n        return subprocess.run([self.binary, \'--config\', self.config] + list(args)\n                              + [\'--retries\', \'2\', \'--low-level-retries\', \'3\',\n                                 \'--contimeout\', \'15s\', \'--timeout\', \'60s\'],\n                              capture_output=True, text=True)\n\n    def exists(self, relative):\n        result = self._call([\'lsjson\', self.root + \'/\' + relative, \'--stat\'])\n        if result.returncode == 0:\n            return True\n        if result.returncode in {3, 4}:\n            return False\n        raise RuntimeError(\'Falha ao consultar Drive (código \' + str(result.returncode) + \'): \' + relative)\n\n    def fetch(self, relative, target, expected_sha=None, optional=False):\n        target = Path(target)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        receipt = target.with_name(target.name + \'.download.json\')\n        if target.is_file() and target.stat().st_size:\n            local_sha = core.sha256_file(target)\n            prior = {}\n            if receipt.is_file():\n                try:\n                    prior = json.loads(receipt.read_text())\n                except (ValueError, OSError):\n                    pass\n            if ((expected_sha and local_sha == expected_sha)\n                    or (not expected_sha and prior.get(\'origem\') == relative and prior.get(\'sha256\') == local_sha)):\n                print(\'Entrada local conferida:\', relative, flush=True)\n                return target\n        if optional and not self.exists(relative):\n            return None\n        print(\'Baixando:\', relative, flush=True)\n        tmp = target.with_name(target.name + \'.part\')\n        result = self._call([\'copyto\', self.root + \'/\' + relative, str(tmp)])\n        if result.returncode:\n            raise RuntimeError(\'Download interrompido (código \' + str(result.returncode) + \'): \' + relative)\n        if not tmp.is_file() or not tmp.stat().st_size:\n            raise RuntimeError(\'Download vazio: \' + relative)\n        actual = core.sha256_file(tmp)\n        if expected_sha and actual != expected_sha:\n            tmp.replace(target.with_name(target.name + \'.rejeitado_\' + actual[:12]))\n            raise DigestMismatch(\'SHA-256 diferente da fonte autenticada: \' + relative)\n        tmp.replace(target)\n        core.save_json(receipt, {\'origem\': relative, \'sha256\': actual, \'bytes\': target.stat().st_size})\n        return target\n\n    def backup(self, local, relative, required=True):\n        print(\'Salvando resultados no Drive...\', flush=True)\n        result = self._call([\'copy\', str(local), self.root + \'/\' + relative,\n                             \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\', \'--exclude\', \'**/__pycache__/**\'])\n        if result.returncode and required:\n            raise RuntimeError(\'Falha no salvamento no Drive; resultados locais preservados. Código \' + str(result.returncode))\n        return result.returncode == 0\n\n    def restore(self, relative, local):\n        if not self.exists(relative):\n            return False\n        result = self._call([\'copy\', self.root + \'/\' + relative, str(local),\n                             \'--exclude\', \'*.zip\', \'--exclude\', \'**/*.tmp\', \'--exclude\', \'**/*.part\'])\n        if result.returncode:\n            raise RuntimeError(\'Não foi possível recuperar o resultado anterior; código \' + str(result.returncode))\n        return True\n', 'final20_eval.py': '"""Inferência com retomada e avaliação explícita da reserva final autorizada."""\nfrom __future__ import annotations\nimport gzip,json\nfrom pathlib import Path\nimport diag11_core as core\nimport conf14_evaluation as e14\nfrom final20_data import COHORT,ROLE,MODELS,digest,save_gz\n\ndef validate_records(records,plan):\n    if len(records)!=plan[\'images\'] or len({r[\'id\'] for r in records})!=len(records):\n        raise ValueError(\'Avaliação exige todos os IDs únicos da reserva.\')\n    if any(r[\'role\']!=ROLE or r[\'coorte\']!=COHORT for r in records):raise ValueError(\'Papel proibido nesta avaliação final.\')\n\ndef predict(model,record,device,plan):\n    if record[\'role\']!=ROLE or record[\'coorte\']!=COHORT:raise ValueError(\'Inferência somente na reserva autorizada.\')\n    args={k:v for k,v in plan[\'inference\'].items() if k!=\'ultralytics\'}\n    result=model.predict(str(record[\'image\']),**args,device=device,verbose=False,save=False)[0]\n    if Path(result.path).resolve()!=Path(record[\'image\']).resolve() or tuple(result.orig_shape)!=(record[\'height\'],record[\'width\']):\n        raise ValueError(\'O modelo retornou uma imagem ou dimensão diferente da entrada.\')\n    boxes=result.boxes\n    if boxes is None:return []\n    return [{\'classe_id\':int(c),\'classe\':core.NAMES[int(c)],\'bbox\':[float(x) for x in box],\'confianca\':float(score)}\n            for c,box,score in zip(boxes.cls.cpu().numpy(),boxes.xyxy.cpu().numpy(),boxes.conf.cpu().numpy())]\n\ndef cache_request(records,model_name,weight_sha,environment,plan):\n    validate_records(records,plan)\n    if model_name not in MODELS or weight_sha!=plan[\'weight_sha256\'][model_name]:raise ValueError(\'Peso fora do protocolo final.\')\n    signatures=[{k:r[k] for k in [\'id\',\'sha256\',\'label_sha256\',\'coverage_sha256\',\'width\',\'height\',\'known_mask\',\'role\']} for r in records]\n    return {\'versao\':\'20-cache-reserva-v1\',\'modelo\':model_name,\'peso_sha256\':weight_sha,\'parametros\':plan[\'inference\'],\n        \'protocolo_sha256\':digest(plan),\'ambiente\':environment,\'entradas_sha256\':digest(signatures),\'imagens\':len(records)}\n\ndef obtain_cache(model,records,model_name,weight_sha,environment,device,plan,directory,checkpoint=None,predict_fn=None):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    request=cache_request(records,model_name,weight_sha,environment,plan);request_sha=digest(request)\n    block_size=25;cache={};reused=0;generated=0;geometry=[]\n    for start in range(0,len(records),block_size):\n        block=records[start:start+block_size];path=directory/f\'lote_{start//block_size:04d}.json.gz\'\n        if path.is_file():\n            payload=json.loads(gzip.decompress(path.read_bytes()))\n            if payload.get(\'request_sha256\')!=request_sha or payload.get(\'rows_sha256\')!=digest(payload.get(\'rows\')):\n                raise ValueError(\'Lote de previsões alterado ou de outro contexto: \'+path.name)\n            values=payload[\'rows\'];reused+=len(values)\n            if [v[\'id\'] for v in values]!=[r[\'id\'] for r in block]:raise ValueError(\'Lote com IDs ou ordem incompatíveis.\')\n        else:\n            values=[]\n            for record in block:\n                predictions=(predict_fn or predict)(model,record,device,plan)\n                e14.validate_predictions(predictions,record)\n                if len(predictions)>plan[\'inference\'][\'max_det\']:raise ValueError(\'Cache excede max_det congelado.\')\n                values.append({\'id\':record[\'id\'],\'image_sha256\':record[\'sha256\'],\'label_sha256\':record[\'label_sha256\'],\n                    \'coverage_sha256\':record[\'coverage_sha256\'],\'predicoes\':predictions})\n            save_gz(path,{\'request_sha256\':request_sha,\'rows_sha256\':digest(values),\'rows\':values});generated+=len(values)\n        for value,record in zip(values,block):\n            if any(value[key]!=record[field] for key,field in [(\'image_sha256\',\'sha256\'),(\'label_sha256\',\'label_sha256\'),(\'coverage_sha256\',\'coverage_sha256\')]):\n                raise ValueError(\'Previsões pertencem a outra imagem ou referência.\')\n            diagnostic=e14.validate_predictions(value[\'predicoes\'],record)\n            geometry.append({\'id\':record[\'id\'],\'diagnostico\':diagnostic})\n            if len(value[\'predicoes\'])>plan[\'inference\'][\'max_det\']:raise ValueError(\'Cache excede max_det.\')\n            cache[record[\'id\']]=value[\'predicoes\']\n        done=start+len(block)\n        core.save_json(directory/\'progresso_cache.json\',{\'request_sha256\':request_sha,\'imagens_concluidas\':done,\'imagens_previstas\':len(records)})\n        print(f\'{model_name}: {done}/{len(records)} imagens; {reused} reutilizadas\',flush=True)\n        if checkpoint and (done%100==0 or done==len(records)):checkpoint()\n    if set(cache)!={r[\'id\'] for r in records}:raise ValueError(\'Cache final incompleto.\')\n    unusual=[v for v in geometry if v[\'diagnostico\'][\'area_zero\'] or v[\'diagnostico\'][\'fora_da_imagem\']]\n    counts={k:sum(v[\'diagnostico\'][k] for v in geometry) for k in [\'predicoes\',\'area_zero\',\'fora_da_imagem\']}\n    geom={\'propostas_removidas\':0,\'coordenadas_alteradas\':False,\'totais\':counts,\'casos\':unusual,\n        \'imagens_atingiram_max_det\':sum(len(v)==plan[\'inference\'][\'max_det\'] for v in cache.values())}\n    core.save_json(directory/\'diagnostico_geometria.json\',geom)\n    meta={\'solicitacao\':request,\'request_sha256\':request_sha,\'concluido\':True,\'imagens\':len(cache),\n        \'imagens_reutilizadas_nesta_execucao\':reused,\'imagens_inferidas_nesta_execucao\':generated,\n        \'limite\':\'Propostas pós-NMS em conf=0,001 e max_det=300; sem logits brutos.\',\n        \'arquivos\':[{ \'arquivo\':p.name,\'sha256\':core.sha256_file(p)} for p in sorted(directory.glob(\'lote_*.json.gz\'))]}\n    core.save_json(directory/\'cache_manifesto.json\',meta)\n    return cache,meta\n\ndef analyse(record,predictions,model_name,plan,policy,matcher):\n    if record[\'role\']!=ROLE or record[\'coorte\']!=COHORT:raise ValueError(\'Avaliação somente da reserva final.\')\n    known=record[\'known_mask\'];conf=plan[\'alert_confidence\']\n    gt=core.with_policy([g for g in record[\'gt\'] if known&(1<<g[\'classe_id\'])],record[\'width\'],record[\'height\'],policy)\n    low=[p for p in predictions if known&(1<<p[\'classe_id\'])]\n    all_pred=core.with_policy([p for p in predictions if p[\'confianca\']>=conf],record[\'width\'],record[\'height\'],policy)\n    pred=[p for p in all_pred if known&(1<<p[\'classe_id\'])]\n    pairs,missing,extras=matcher(gt,pred)\n    events=[e14.event(record,\'correspondencia\',gt[gi],pred[pi],float(overlap),low,conf) for gi,pi,overlap in pairs]\n    events += [e14.event(record,\'objeto_nao_detectado\',gt[gi],None,None,low,conf) for gi in missing]\n    events += [e14.event(record,\'previsao_nao_associada\',None,pred[pi],None,low,conf) for pi in extras]\n    for event in events:event[\'modelo\']=model_name\n    messages=[]\n    for obj in all_pred:\n        if core.alert_on(obj):\n            message=policy.gerar_mensagem(obj,\'medio\')\n            if not known&(1<<obj[\'classe_id\']):message+=\' Classe sem cobertura de referência nesta imagem.\'\n            messages.append(message)\n    if any(not isinstance(m,str) or not m.strip() for m in messages):raise ValueError(\'Alerta sem mensagem.\')\n    image={\'coorte\':COHORT,\'imagem\':record[\'name\'],\'id\':record[\'id\'],\'modelo\':model_name,\'limiar_confianca\':conf,\n        \'tp_alerta\':sum(e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'fn_alerta\':sum(e[\'ha_alerta_ref\'] and not e[\'ha_alerta_pred\'] for e in events),\n        \'fp_alerta\':sum(not e[\'ha_alerta_ref\'] and e[\'ha_alerta_pred\'] for e in events),\n        \'alta_ref\':sum(g[\'prioridade\']==\'alto\' for g in gt),\n        \'alta_sem_alerta\':sum(e[\'prioridade_ref\']==\'alto\' and not e[\'ha_alerta_pred\'] for e in events),\n        \'tp_detector\':len(pairs),\'fn_detector\':len(missing),\'fp_detector\':len(extras),\n        \'gt_avaliaveis\':len(gt),\'gt_ignorados\':len(record[\'gt\'])-len(gt),\n        \'predicoes_ignoradas_conf\':len(all_pred)-len(pred),\n        \'cobertura_completa_alertas\':known&core.ALERT_MASK==core.ALERT_MASK,\n        \'ha_alerta_ref\':any(core.alert_on(g) for g in gt),\'ha_alerta_pred\':any(core.alert_on(p) for p in pred),\n        \'mensagem_alerta\':\'\\n\'.join(messages)}\n    return image,events\n\ndef evaluate(records,cache,model_name,plan,policy,matcher,output):\n    validate_records(records,plan)\n    if model_name not in MODELS or set(cache)!={r[\'id\'] for r in records}:raise ValueError(\'Modelo ou IDs fora do protocolo.\')\n    images=[];events=[]\n    for record in records:\n        e14.validate_predictions(cache[record[\'id\']],record)\n        image,part=analyse(record,cache[record[\'id\']],model_name,plan,policy,matcher)\n        images.append(image);events.extend(part)\n    summary,classes=e14.summarise(images,events,plan[\'alert_confidence\'])\n    summary.update(modelo=model_name,papel=ROLE,reserva_final=True,treinamento_executado=False)\n    summary[\'predicoes_sem_cobertura_no_limiar\']=summary[\'predicoes_ignoradas_conf\']\n    for row in classes:\n        cid=row[\'classe_id\'];part=[e for e in events if e[\'classe_id\']==cid]\n        row[\'taxa_alta_sem_alerta\']=core.ratio(row[\'alta_sem_alerta\'],row[\'alta_ref\'])\n        row[\'imagens_com_cobertura\']=sum(bool(r[\'known_mask\']&(1<<cid)) for r in records)\n        detector=core.alert_metrics(sum(e[\'evento\']==\'correspondencia\' for e in part),\n            sum(e[\'evento\']==\'objeto_nao_detectado\' for e in part),sum(e[\'evento\']==\'previsao_nao_associada\' for e in part))\n        row.update({\'detector_\'+k:v for k,v in detector.items()})\n    if summary[\'alertas_por_objeto\'][\'tp\']+summary[\'alertas_por_objeto\'][\'fn\']!=sum(core.alert_on(g) for r in records for g in core.with_policy([g for g in r[\'gt\'] if r[\'known_mask\']&(1<<g[\'classe_id\'])],r[\'width\'],r[\'height\'],policy)):\n        raise ValueError(\'Denominador de alertas GT inconsistente.\')\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    core.save_json(output/\'metricas.json\',summary)\n    core.save_csv(output/\'metricas_por_classe.csv\',classes)\n    core.save_csv(output/\'alertas_por_imagem.csv\',images)\n    core.save_csv(output/\'avaliacao_objetos.csv\',events)\n    result={\'summary\':summary,\'classes\':classes,\'images\':images,\'events\':events}\n    save_gz(output/\'avaliacao_snapshot.json.gz\',result)\n    return result\n\ndef run_model(weight,records,model_name,environment,device,plan,policy,matcher,output,checkpoint=None,model_factory=None,predict_fn=None):\n    weight=Path(weight)\n    if core.sha256_file(weight)!=plan[\'weight_sha256\'][model_name]:raise ValueError(\'Peso diferente do congelado.\')\n    validate_records(records,plan)\n    if model_factory is None:\n        from ultralytics import YOLO\n        model_factory=YOLO\n    model=model_factory(str(weight))\n    try:\n        names=model.names if isinstance(model.names,dict) else dict(enumerate(model.names))\n        if names!=dict(enumerate(core.NAMES)):raise ValueError(\'Mapeamento de classes do peso incompatível.\')\n        cache,meta=obtain_cache(model,records,model_name,plan[\'weight_sha256\'][model_name],environment,device,plan,\n            Path(output)/\'cache\',checkpoint,predict_fn)\n        result=evaluate(records,cache,model_name,plan,policy,matcher,output)\n        result[\'cache_manifest\']=meta\n        return result\n    finally:\n        del model\n', 'final20_io.py': '"""Pacote final e estado de execução, sem pesos, imagens ou credenciais."""\nfrom __future__ import annotations\nfrom datetime import datetime,timezone\nimport json,zipfile\nfrom pathlib import Path\nimport diag11_core as core\n\ndef now():return datetime.now(timezone.utc).isoformat()\n\ndef state(output,**changes):\n    path=Path(output)/\'estado_execucao_20.json\'\n    previous=json.loads(path.read_text()) if path.is_file() else {}\n    previous.update(changes);previous[\'atualizado_em_utc\']=now();core.save_json(path,previous)\n    return previous\n\ndef package(output):\n    output=Path(output);target=output/\'pacote_avaliacao_final_20.zip\';tmp=target.with_name(target.name+\'.tmp\')\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=4) as z:\n        for p in sorted(output.rglob(\'*\')):\n            if not p.is_file() or p in {target,tmp} or p.name==\'salvamento_drive_20.json\' or p.suffix in {\'.pt\',\'.tmp\',\'.part\',\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'}:continue\n            rel=p.relative_to(output)\n            if \'__pycache__\' in rel.parts:continue\n            z.write(p,rel.as_posix())\n    tmp.replace(target);return target\n\ndef manifest(output):\n    output=Path(output);excluded={\'integridade_arquivos_20.json\',\'pacote_avaliacao_final_20.zip\',\'salvamento_drive_20.json\'}\n    rows=[]\n    for p in sorted(output.rglob(\'*\')):\n        if p.is_file() and p.name not in excluded and p.suffix not in {\'.pt\',\'.tmp\',\'.part\',\'.jpg\',\'.jpeg\',\'.png\',\'.webp\',\'.bmp\'} and \'__pycache__\' not in p.parts:\n            rows.append({\'arquivo\':p.relative_to(output).as_posix(),\'bytes\':p.stat().st_size,\'sha256\':core.sha256_file(p)})\n    core.save_json(output/\'integridade_arquivos_20.json\',{\'arquivos\':rows,\'gerado_em_utc\':now()})\n    return rows\n\ndef backup(drive,output,remote):\n    try:ok=drive.backup(output,remote,required=False)\n    except Exception:ok=False\n    if not ok:print(\'Cópia no Drive indisponível neste momento; arquivos locais preservados.\',flush=True)\n    return ok\n\ndef failure(output,error,stage,drive=None,remote=None):\n    output=Path(output)\n    status=state(output,concluido=False,etapa=stage,erro={\'tipo\':type(error).__name__,\'mensagem\':str(error)[:1400]})\n    core.save_json(output/\'execucao_interrompida_20.json\',status)\n    core.save_json(output/\'resumo_avaliacao_final_20.json\',{**status,\'status\':\'execucao_interrompida\',\'pacote_parcial\':True})\n    manifest(output)\n    result=package(output)\n    if drive and remote:backup(drive,output,remote)\n    return result\n', 'final20_report.py': '"""Relatório final, contagens e texto-base sem seleção adaptativa de modelos."""\nfrom __future__ import annotations\nimport html,json\nfrom pathlib import Path\nimport diag11_core as core\nfrom final20_data import MODELS,digest\n\ndef pct(v):return \'indefinido (sem denominador)\' if v is None else f\'{100*v:.2f}%\'.replace(\'.\',\',\')\ndef pp(v):return \'indefinido\' if v is None else f\'{100*v:+.2f} p.p.\'.replace(\'.\',\',\')\ndef ci(row,delta=False):\n    if row[\'ic95_inferior\'] is None:return \'IC indefinido\'\n    format_value=pp if delta else pct\n    return f"[{format_value(row[\'ic95_inferior\'])}; {format_value(row[\'ic95_superior\'])}]"\ndef table(rows):\n    if not rows:return \'<p>Sem registros.</p>\'\n    keys=list(rows[0])\n    return \'<div class="scroll"><table><thead><tr>\'+\'\'.join(\'<th>\'+html.escape(str(k))+\'</th>\' for k in keys)+\'</tr></thead><tbody>\'+\'\'.join(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(r.get(k,\'\')))+\'</td>\' for k in keys)+\'</tr>\' for r in rows)+\'</tbody></table></div>\'\n\ndef build(records,evaluations,intervals,boot,groups,extraction,plan,expected,context,output):\n    output=Path(output);find=lambda model,metric:next(r for r in intervals if r[\'escopo\']==\'global\' and r[\'modelo_ou_comparacao\']==model and r[\'metrica\']==metric)\n    rows=[];shown=[];classes=[]\n    for model in MODELS:\n        result=evaluations[model];s=result[\'summary\'];a=s[\'alertas_por_objeto\']\n        if s[\'imagens\']!=plan[\'images\'] or not result[\'cache_manifest\'][\'concluido\']:raise ValueError(\'Resultado final incompleto.\')\n        row={\'modelo\':model,\'imagens\':s[\'imagens\'],**a,\'alertas_GT\':a[\'tp\']+a[\'fn\'],\'alertas_pred_avaliaveis\':a[\'tp\']+a[\'fp\'],\n            \'omissoes_altas\':s[\'alta_sem_alerta\'],\'GT_prioridade_alta\':s[\'alta_ref\'],\'taxa_omissoes_altas\':s[\'taxa_alta_sem_alerta\'],\n            \'GT_sem_cobertura\':s[\'gt_ignorados_sem_cobertura\'],\'previsoes_sem_cobertura_no_limiar\':s[\'predicoes_sem_cobertura_no_limiar\']}\n        for metric in [\'precision\',\'recall\',\'f1\',\'taxa_alta_sem_alerta\']:\n            interval=find(model,metric)\n            for k in [\'ic95_inferior\',\'ic95_superior\',\'replicas_validas\']:row[metric+\'_\'+k]=interval[k]\n        rows.append(row)\n        shown.append({\'Modelo\':model,\'Precisão\':pct(a[\'precision\']),\'Recall\':pct(a[\'recall\']),\'F1\':pct(a[\'f1\']),\n            \'TP / FN / FP\':f"{a[\'tp\']} / {a[\'fn\']} / {a[\'fp\']}",\n            \'Omissões altas / GT alto\':f"{s[\'alta_sem_alerta\']} / {s[\'alta_ref\']}",\'Taxa omissões altas\':pct(s[\'taxa_alta_sem_alerta\'])})\n        classes += [{\'modelo\':model,**r} for r in result[\'classes\']]\n    if rows[0][\'alertas_GT\']!=rows[1][\'alertas_GT\'] or rows[0][\'GT_prioridade_alta\']!=rows[1][\'GT_prioridade_alta\']:\n        raise ValueError(\'Modelos avaliados com denominadores GT diferentes.\')\n    core.save_csv(output/\'comparacao_final_20.csv\',rows)\n    core.save_csv(output/\'resultados_por_classe_20.csv\',classes)\n    delta=find(\'delta_candidato10_menos_baseline\',\'recall\');primary=find(\'baseline\',\'recall\')\n    primary_rows=[r for r in intervals if r[\'estimando_primario\']]\n    core.save_csv(output/\'estimandos_primarios_20.csv\',primary_rows)\n    limitations=list(plan[\'limits\'])+[\n        \'Cobertura parcial: previsões de classes sem anotação não são julgadas como acertos ou erros.\',\n        \'A reserva é uma amostra planejada para diversidade de classes; não estima automaticamente a frequência de obstáculos no uso cotidiano.\',\n        \'As validações Home e externa de desenvolvimento já influenciaram os ajustes e são apresentadas separadamente.\',\n        \'Classes com menos de 30 GT são sinalizadas; métricas sem denominador são indefinidas.\',\n        \'A política V2 usa classe, posição e tamanho aparente; as métricas de alerta medem concordância com essa regra.\'\n    ]\n    if extraction[\'pares_semelhanca_phash_com_desenvolvimento\']:\n        limitations.append(\'Há semelhanças pHash com imagens de desenvolvimento registradas em extracao_reserva_20.json; interpretar a separação com essa ressalva. Nenhuma imagem foi substituída.\')\n    summary={\'version\':\'20-avaliacao-final-1.0.0\',\'concluido\':True,\'status\':\'avaliacao_final_concluida\',\n        \'fase_desenvolvimento_encerrada\':True,\'context_key\':context,\'pacote_fonte_19_sha256\':expected[\'pacote_19\'],\n        \'protocolo_sha256\':digest(plan),\'dataset_sha256\':plan[\'dataset_sha256\'],\'modelos\':list(MODELS),\n        \'pesos_sha256\':plan[\'weight_sha256\'],\'modelo_principal_predefinido\':\'baseline\',\'comparador_predefinido\':\'candidato_10\',\n        \'reserva_imagens\':len(records),\'reserva_avaliada\':True,\'reserva_inferida\':True,\n        \'treinamento_executado\':False,\'pesos_ou_limiares_ajustados\':False,\'modelo_substituido\':False,\n        \'test_home_inferido\':False,\'test_oi20k_inferido\':False,\'preenchimento_manual_obrigatorio\':False,\n        \'estimandos_primarios\':primary_rows,\'bootstrap\':boot,\'grupos\':groups[\'grupos\'],\n        \'resultados\':rows,\'limites\':limitations,\'decisao\':\'Encerrar a fase experimental e relatar metodologia, resultados e limitações; sem nova seleção de pesos na reserva.\'}\n    core.save_json(output/\'resumo_avaliacao_final_20.json\',summary)\n    text=\'# Avaliação final dos modelos e encerramento experimental\\n\\n\'\n    text+=f"A avaliação final utilizou os {len(records)} IDs previamente reservados do conjunto externo, com {groups[\'grupos\']} grupos de imagens para a análise de incerteza. Foram avaliados o baseline, definido como principal, e o candidato 10, definido como comparador, segundo o protocolo fixado no notebook 19 antes da consulta à reserva. Não houve treinamento ou ajuste de pesos e limiares nesta etapa.\\n\\n"\n    text+=\'A política V2, a confiança de 0,25 e a associação por classe com IoU mínimo de 0,5 foram mantidas. A captura usou Ultralytics 8.4.150, confiança 0,001, entrada 640, NMS IoU 0,70, limite de 300 propostas, batch 1 e FP32. As métricas de alertas medem concordância com a política aplicada às caixas anotadas nas classes com cobertura.\\n\\n\'\n    text+=\'| Modelo | Precisão | Recall | F1 | TP | FN | FP | Omissões altas / GT alto |\\n| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |\\n\'\n    for r in rows:\n        text+=f"| {r[\'modelo\']} | {pct(r[\'precision\'])} | {pct(r[\'recall\'])} | {pct(r[\'f1\'])} | {r[\'tp\']} | {r[\'fn\']} | {r[\'fp\']} | {r[\'omissoes_altas\']} / {r[\'GT_prioridade_alta\']} |\\n"\n    text+=f"\\nO recall do baseline foi {pct(primary[\'estimativa\'])}, com IC de 95% {ci(primary)}. A diferença pareada de recall, candidato 10 menos baseline, foi {pp(delta[\'estimativa\'])}, com IC de 95% {ci(delta,True)}. Esse intervalo descreve a incerteza da diferença neste protocolo, sem estabelecer superioridade universal ou autorizar substituição automática do modelo.\\n\\n"\n    text+=f"Os intervalos utilizaram 2.000 reamostragens pareadas de grupos, com reposição e semente {plan[\'confidence_intervals\'][\'seed\']}. Imagens com bytes iguais ou pHash64 com distância de Hamming até 4 integram o mesmo componente transitivo. Cada réplica utilizou os mesmos grupos para ambos os modelos. As métricas foram calculadas como razões das contagens agregadas, mantendo réplicas sem denominador como indefinidas. No recall principal do baseline e na diferença de recall, houve {primary[\'replicas_validas\']} e {delta[\'replicas_validas\']} réplicas válidas, respectivamente.\\n\\n"\n    text+=\'Os resultados por classe, os suportes e os intervalos completos estão nas tabelas anexas. As 12 classes permanecem documentadas, inclusive aquelas que não geram alertas pela V2. As métricas de detecção no limiar de 0,25 também são fornecidas separadamente das métricas de alerta. Os valores absolutos de omissões da validação de 1.000 imagens não foram usados como limites de aprovação para a reserva de 300.\\n\\n\'\n    text+=\'A independência se refere à reserva preservada no fluxo documentado. O histórico completo do treinamento original do baseline não está comprovado e não permite garantir ausência de exposição em todo pré-treino. A amostra também não representa automaticamente a distribuição de cenas do uso cotidiano. A avaliação não mede distância real ou segurança de locomoção.\\n\\n\'\n    text+=\'A fase experimental foi encerrada com a caracterização dos dois modelos predefinidos. Os resultados de desenvolvimento e os resultados finais devem ser apresentados em tabelas separadas. A continuação do trabalho consiste em redigir a metodologia, discutir os resultados e limitações e utilizar a demonstração do notebook 19 como material ilustrativo.\\n\'\n    (output/\'texto_base_resultados_finais_20.md\').write_text(text,encoding=\'utf-8\')\n    main_intervals=[]\n    for model in [*MODELS,\'delta_candidato10_menos_baseline\']:\n        for metric in [\'precision\',\'recall\',\'f1\',\'taxa_alta_sem_alerta\']:\n            row=find(model,metric);isdelta=model.startswith(\'delta_\')\n            main_intervals.append({\'Modelo / comparação\':model,\'Métrica\':metric,\'Estimativa\':pp(row[\'estimativa\']) if isdelta else pct(row[\'estimativa\']),\n                \'IC 95%\':ci(row,isdelta),\'Réplicas válidas\':f"{row[\'replicas_validas\']}/2000",\'Primário\':\'sim\' if row[\'estimando_primario\'] else \'secundário\'})\n    class_shown=[{\'Modelo\':r[\'modelo\'],\'Classe\':r[\'classe\'],\'GT\':r[\'gt_avaliaveis\'],\'Imagens com cobertura\':r[\'imagens_com_cobertura\'],\n        \'Suporte <30 GT\':\'sim\' if r[\'suporte_reduzido\'] else \'não\',\'Precisão alertas\':pct(r[\'precision\']),\'Recall alertas\':pct(r[\'recall\']),\n        \'Precisão detector\':pct(r[\'detector_precision\']),\'Recall detector\':pct(r[\'detector_recall\'])} for r in classes]\n    page=\'\'\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Avaliação final 20</title><style>body{font:16px/1.6 system-ui;background:#eef2f6;color:#162739;margin:0}main{max-width:1450px;margin:auto;padding:28px}.note{padding:20px;background:white;border:1px solid #cbd5e1;border-radius:12px;margin:20px 0}.scroll{overflow:auto}table{width:100%;border-collapse:collapse;background:white}th,td{padding:10px;border:1px solid #d1dae4;text-align:left}h1,h2{line-height:1.25}a{color:#1d4ed8}</style><main><p>TCC · Encerramento experimental</p><h1>Avaliação final na reserva externa</h1>\'\'\'\n    page+=f\'<div class="note"><strong>{len(records)} imagens · {groups["grupos"]} grupos · dois modelos predefinidos</strong><p>Protocolo fixado no notebook 19. Confiança 0,25, política V2, IoU de associação 0,5. Nenhum treinamento ou ajuste após a consulta à reserva.</p><p>Fase experimental concluída. Os dois modelos foram caracterizados sem seleção automática para uso.</p></div>\'\n    page+=\'<h2>Resultados finais dos alertas</h2>\'+table(shown)\n    page+=\'<h2>Incerteza e comparação pareada</h2><p>Bootstrap de 2.000 réplicas com os mesmos grupos para ambos os modelos; intervalos percentis de 95%. Diferenças em pontos percentuais. Denominadores nulos permanecem indefinidos.</p>\'+table(main_intervals)\n    page+=\'<h2>Resultados por classe e cobertura</h2><p>As métricas consideram somente classes com referência na imagem. Classes sem cobertura permanecem sem julgamento. Os intervalos por classe estão no CSV completo. Resultados secundários são descritivos.</p>\'+table(class_shown)\n    page+=\'<h2>Limitações</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(v)+\'</li>\' for v in limitations)+\'</ul>\'\n    page+=\'<h2>Arquivos para o relato</h2><ul>\'+\'\'.join(f\'<li><a href="{n}">{n}</a></li>\' for n in [\'comparacao_final_20.csv\',\'estimandos_primarios_20.csv\',\'resultados_por_classe_20.csv\',\'intervalos_confianca_20.csv\',\'texto_base_resultados_finais_20.md\',\'fontes19/comparacao_finais_e_referencias.csv\',\'fontes19/protocolo_avaliacao_final.json\'])+\'</ul></main></html>\'\n    (output/\'relatorio_avaliacao_final_20.html\').write_text(page,encoding=\'utf-8\')\n    (output/\'LEIA_RESULTADO.md\').write_text(\'# Avaliação final 20 concluída\\n\\nAbra relatorio_avaliacao_final_20.html para os resultados. O texto-base está em texto_base_resultados_finais_20.md. As tabelas principais são comparacao_final_20.csv, resultados_por_classe_20.csv e intervalos_confianca_20.csv.\\n\\nResultados em pacote_avaliacao_final_20.zip para a análise e a escrita do TCC. Não ajuste os modelos ou limiares usando esta reserva. Nenhum novo treinamento é necessário para encerrar esta fase.\\n\',encoding=\'utf-8\')\n    return summary\n', 'final20_stats.py': '"""Bootstrap pareado dos grupos, com denominadores indefinidos preservados."""\nfrom __future__ import annotations\nfrom collections import Counter,defaultdict\nfrom pathlib import Path\nimport numpy as np\nimport diag11_core as core\nfrom final20_data import MODELS,digest,save_gz\n\nMETRICS=(\'precision\',\'recall\',\'f1\',\'taxa_alta_sem_alerta\')\nSCOPES=(\'global\',)+tuple(core.NAMES)\n\ndef make_groups(records,output=None):\n    ordered=sorted(records,key=lambda r:r[\'id\'])\n    if len({r[\'id\'] for r in ordered})!=len(ordered):raise ValueError(\'IDs repetidos na formação dos grupos.\')\n    parent=list(range(len(ordered)))\n    def find(x):\n        while parent[x]!=x:parent[x]=parent[parent[x]];x=parent[x]\n        return x\n    def union(a,b):\n        a,b=find(a),find(b)\n        if a!=b:parent[max(a,b)]=min(a,b)\n    links=[]\n    for i,a in enumerate(ordered):\n        for j in range(i):\n            b=ordered[j];distance=(int(a[\'phash\'])^int(b[\'phash\'])).bit_count()\n            if a[\'sha256\']==b[\'sha256\'] or distance<=4:\n                union(i,j);links.append({\'a\':a[\'id\'],\'b\':b[\'id\'],\'bytes_iguais\':a[\'sha256\']==b[\'sha256\'],\'distancia_phash\':distance})\n    buckets=defaultdict(list)\n    for i,r in enumerate(ordered):buckets[find(i)].append(r[\'id\'])\n    groups=sorted((sorted(ids) for ids in buckets.values()),key=lambda ids:ids[0])\n    mapping={iid:digest(ids)[:24] for ids in groups for iid in ids}\n    report={\'metodo\':\'SHA-256 igual ou pHash64 Hamming <=4; componentes transitivos\',\n        \'phash_origem\':\'valores congelados no manifesto autenticado, antes desta avaliação\',\n        \'imagens\':len(ordered),\'grupos\':len(groups),\'distribuicao_tamanhos\':dict(sorted(Counter(map(len,groups)).items())),\n        \'maior_grupo\':max(map(len,groups),default=0),\'componentes\':[{\'grupo\':digest(ids)[:24],\'ids\':ids} for ids in groups],\n        \'ligacoes\':links,\'grupos_sha256\':digest(groups)}\n    if output is not None:\n        output=Path(output);core.save_json(output/\'grupos_bootstrap_20.json\',report)\n        core.save_csv(output/\'grupos_por_imagem.csv\',[{\'id\':r[\'id\'],\'grupo\':mapping[r[\'id\']],\'sha256\':r[\'sha256\'],\'phash\':f\'{r["phash"]:016x}\'} for r in ordered])\n    return groups,mapping,report\n\ndef rates(counts):\n    c=np.asarray(counts,dtype=float)\n    tp,fn,fp,high_missing,high_total=[c[...,i] for i in range(5)]\n    denominators=[tp+fp,tp+fn,2*tp+fp+fn,high_total]\n    numerators=[tp,tp,2*tp,high_missing]\n    result=[]\n    for n,d in zip(numerators,denominators):\n        result.append(np.divide(n,d,out=np.full_like(d,np.nan,dtype=float),where=d>0))\n    return np.stack(result,axis=-1)\n\ndef build_counts(records,evaluations):\n    ids=[r[\'id\'] for r in records];positions={iid:i for i,iid in enumerate(ids)}\n    counts=np.zeros((2,len(records),len(SCOPES),5),dtype=np.int64)\n    supports=np.zeros(len(SCOPES),dtype=np.int64)\n    for r in records:\n        for g in r[\'gt\']:\n            if r[\'known_mask\']&(1<<g[\'classe_id\']):supports[0]+=1;supports[1+g[\'classe_id\']]+=1\n    for mi,model in enumerate(MODELS):\n        evaluated=evaluations[model]\n        if {v[\'id\'] for v in evaluated[\'images\']}!=set(ids) or len(evaluated[\'images\'])!=len(ids):raise ValueError(\'Amostras diferentes no pareamento.\')\n        for event in evaluated[\'events\']:\n            if event[\'id\'] not in positions:raise ValueError(\'Evento fora da reserva.\')\n            vector=np.asarray([event[\'ha_alerta_ref\'] and event[\'ha_alerta_pred\'],\n                event[\'ha_alerta_ref\'] and not event[\'ha_alerta_pred\'],\n                not event[\'ha_alerta_ref\'] and event[\'ha_alerta_pred\'],\n                event[\'prioridade_ref\']==\'alto\' and not event[\'ha_alerta_pred\'],event[\'prioridade_ref\']==\'alto\'],dtype=np.int64)\n            i=positions[event[\'id\']];counts[mi,i,0]+=vector;counts[mi,i,1+event[\'classe_id\']]+=vector\n        point=counts[mi,:,0].sum(axis=0);s=evaluated[\'summary\']\n        if list(point)!=[s[\'alertas_por_objeto\'][\'tp\'],s[\'alertas_por_objeto\'][\'fn\'],s[\'alertas_por_objeto\'][\'fp\'],s[\'alta_sem_alerta\'],s[\'alta_ref\']]:\n            raise ValueError(\'Estatísticas de reamostragem divergem das métricas.\')\n    if not np.array_equal(counts[0,:,:,0]+counts[0,:,:,1],counts[1,:,:,0]+counts[1,:,:,1]) or not np.array_equal(counts[0,:,:,4],counts[1,:,:,4]):\n        raise ValueError(\'Os dois modelos não usam os mesmos denominadores GT.\')\n    return counts,supports\n\ndef bootstrap(records,evaluations,groups,plan,output):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    counts,supports=build_counts(records,evaluations)\n    positions={r[\'id\']:i for i,r in enumerate(records)}\n    flat=[iid for group in groups for iid in group]\n    if set(flat)!=set(positions) or len(flat)!=len(positions) or any(not g for g in groups):raise ValueError(\'Grupos não particionam a reserva.\')\n    grouped=np.stack([counts[:,[positions[iid] for iid in group]].sum(axis=1) for group in groups],axis=0)\n    repetitions=plan[\'confidence_intervals\'][\'repetitions\'];seed=plan[\'confidence_intervals\'][\'seed\']\n    rng=np.random.default_rng(seed);g=len(groups)\n    weights=np.empty((repetitions,g),dtype=np.int32)\n    for i in range(repetitions):weights[i]=np.bincount(rng.integers(0,g,size=g),minlength=g)\n    sampled=np.tensordot(weights,grouped,axes=(1,0)) # réplica, modelo, escopo, contagens\n    sampled_rates=rates(sampled);points=rates(counts.sum(axis=1))\n    differences=sampled_rates[:,1]-sampled_rates[:,0]\n    rows=[]\n    for si,scope in enumerate(SCOPES):\n        total_counts=counts[:,:,si].sum(axis=1)\n        for name,mi in [(MODELS[0],0),(MODELS[1],1),(\'delta_candidato10_menos_baseline\',None)]:\n            for ki,metric in enumerate(METRICS):\n                values=sampled_rates[:,mi,si,ki] if mi is not None else differences[:,si,ki]\n                point=points[mi,si,ki] if mi is not None else points[1,si,ki]-points[0,si,ki]\n                valid=values[np.isfinite(values)]\n                lower,upper=np.percentile(valid,[2.5,97.5]) if len(valid) else (None,None)\n                n,d=(None,None)\n                if mi is not None:\n                    tp,fn,fp,hm,ht=map(int,total_counts[mi]);n=[tp,tp,2*tp,hm][ki];d=[tp+fp,tp+fn,2*tp+fp+fn,ht][ki]\n                rows.append({\'escopo\':scope,\'modelo_ou_comparacao\':name,\'metrica\':metric,\n                    \'estimativa\':float(point) if np.isfinite(point) else None,\n                    \'ic95_inferior\':float(lower) if lower is not None else None,\n                    \'ic95_superior\':float(upper) if upper is not None else None,\n                    \'replicas_validas\':len(valid),\'replicas_previstas\':repetitions,\n                    \'numerador\':n,\'denominador\':d,\'gt_avaliaveis\':int(supports[si]),\n                    \'suporte_reduzido\':bool(supports[si]<30),\n                    \'estimando_primario\':scope==\'global\' and metric==\'recall\' and name in {MODELS[0],\'delta_candidato10_menos_baseline\'},\n                    \'unidade\':\'proporcao\' if mi is not None else \'diferenca_de_proporcoes\'})\n    core.save_csv(output/\'intervalos_confianca_20.csv\',rows)\n    core.save_json(output/\'intervalos_confianca_20.json\',rows)\n    save_gz(output/\'replicas_bootstrap_20.json.gz\',{\'modelos\':MODELS,\'escopos\':SCOPES,\'metricas\':METRICS,\n        \'valores\':sampled_rates.tolist(),\'delta_candidato10_menos_baseline\':differences.tolist()})\n    # NPZ permite reproduzir cada estimativa usando as mesmas multiplicidades de grupos.\n    temp=output/\'reamostragem_grupos_20.tmp.npz\'\n    np.savez_compressed(temp,multiplicidades=weights,contagens_grupos=grouped,contagens_por_imagem=counts)\n    temp.replace(output/\'reamostragem_grupos_20.npz\')\n    report={\'metodo\':plan[\'confidence_intervals\'],\'imagens\':len(records),\'grupos\':g,\'pareado\':True,\n        \'metricas\':list(METRICS),\'escopos\':list(SCOPES),\'total_estimativas\':len(rows),\n        \'replicas_iguais_ambos_modelos\':True,\'denominadores_zero_convertidos_em_zero\':False,\n        \'ordem_grupos_sha256\':digest(groups),\'rng\':\'numpy.random.default_rng; amostra de G grupos com reposição por réplica\',\n        \'analise\':\'Razões de somas dos objetos; sem média das métricas por imagem.\',\n        \'limites\':[\'Intervalos por classe e métricas secundárias são descritivos; sem ajuste por múltiplas comparações.\',\n                  \'Grupos pequenos e poucos objetos podem produzir intervalos instáveis ou degenerados.\']}\n    core.save_json(output/\'bootstrap_resumo_20.json\',report)\n    return rows,report\n', 'src/__init__.py': '', 'src/politica_alertas.py': '"""Parâmetros e funções da V2 original, sem reestimar com o TEST."""\nimport math\n\nNOMES_PT = {"bed": "cama", "sofa": "sofá", "chair": "cadeira", "table": "mesa",\n            "wardrobe": "guarda-roupa", "potted plant": "planta em vaso"}\nVERBOS_PT = {"bed": "detectada", "sofa": "detectado", "chair": "detectada", "table": "detectada",\n             "wardrobe": "detectado", "potted plant": "detectada"}\nORDEM_PRIORIDADE = {"sem_prioridade": 0, "baixo": 1, "medio": 2, "alto": 3}\n\n\ndef avaliar_bbox(classe, bbox, largura, altura, versao="v2"):\n    if versao not in {"v1", "v2"}:\n        raise ValueError("Versão deve ser v1 ou v2.")\n    x1, y1, x2, y2 = map(float, bbox)\n    if largura <= 0 or altura <= 0 or not all(math.isfinite(v) for v in [x1, y1, x2, y2]):\n        raise ValueError("Dimensões ou coordenadas inválidas.")\n    if x2 <= x1 or y2 <= y1:\n        raise ValueError("Caixa sem área positiva.")\n    posicao = classificar_posicao((x1 + x2) / 2, largura)\n    tamanho, area = classificar_tamanho((x2 - x1) * (y2 - y1), largura * altura)\n    vertical, base = classificar_posicao_vertical(y2, altura)\n    resultado = (classificar_risco_v2(classe, posicao, tamanho, vertical) if versao == "v2"\n                 else classificar_risco_v1(classe, posicao, tamanho))\n    nivel = resultado["nivel_risco"]\n    return {"prioridade": nivel if nivel in {"baixo", "medio", "alto"} else "sem_prioridade",\n            "status_original": nivel, "pontuacao": resultado["pontuacao"],\n            "posicao": posicao, "tamanho": tamanho, "regiao_vertical": vertical,\n            "area_relativa": area, "base_vertical_relativa": base,\n            "ajuste_vertical": resultado.get("ajuste_vertical", 0)}\n\n\ndef gerar_mensagem(objeto, nivel_minimo="medio"):\n    if nivel_minimo not in {"medio", "alto"}:\n        raise ValueError("Nível mínimo deve ser medio ou alto.")\n    prioridade = objeto["prioridade"]\n    if ORDEM_PRIORIDADE.get(prioridade, 0) < ORDEM_PRIORIDADE[nivel_minimo]:\n        return ""\n    posicao = {"centro": "central", "esquerda": "esquerda", "direita": "direita"}[objeto["posicao"]]\n    classe = objeto["classe"]\n    return (f"Atenção: {NOMES_PT[classe]} {VERBOS_PT[classe]} na região {posicao} da imagem. "\n            f"Prioridade {\'alta\' if prioridade == \'alto\' else \'média\'}.")\n\n\ndef gerar_mensagens_tabela(df, coluna="nivel_risco_v2"):\n    if df.empty:\n        return []\n    candidatos = df.loc[df[coluna].isin(["medio", "alto"])].copy()\n    candidatos["_ordem"] = candidatos[coluna].map(ORDEM_PRIORIDADE)\n    candidatos = candidatos.sort_values(["_ordem", "ocupacao_imagem"], ascending=[False, False])\n    return [gerar_mensagem({"classe": r["classe"], "posicao": r["posicao"], "prioridade": r[coluna]})\n            for _, r in candidatos.iterrows()]\n\nPESO_OBJETO = {\n    "bed": 3,\n    "sofa": 3,\n    "chair": 3,\n    "table": 3,\n    "wardrobe": 3,\n    "potted plant": 2,\n}\n\nCLASSES_CONTEXTUAIS = {\n    "lamp",\n    "door",\n}\n\nCLASSES_IGNORADAS = {\n    "tv",\n    "laptop",\n    "window",\n    "photo frame",\n}\n\n\nLIMITE_ESQUERDA = 1 / 3\nLIMITE_DIREITA = 2 / 3\n\nLIMITE_AREA_MEDIA = 0.05\nLIMITE_AREA_GRANDE = 0.20\n\nLIMITE_VERTICAL_SUPERIOR = 0.60\nLIMITE_VERTICAL_INFERIOR = 0.80\n\nPESO_POSICAO = {\n    "esquerda": 1,\n    "centro": 3,\n    "direita": 1,\n}\n\nPESO_TAMANHO = {\n    "pequeno": 1,\n    "medio": 2,\n    "grande": 3,\n}\n\nAJUSTE_VERTICAL = {\n    "superior": -2,\n    "intermediaria": -1,\n    "inferior": 0,\n}\n\ndef classificar_posicao(\n    centro_x: float,\n    largura_imagem: int,\n) -> str:\n    centro_normalizado = centro_x / largura_imagem\n\n    if centro_normalizado < LIMITE_ESQUERDA:\n        return "esquerda"\n\n    if centro_normalizado < LIMITE_DIREITA:\n        return "centro"\n\n    return "direita"\n\n\ndef classificar_tamanho(\n    area_box: float,\n    area_imagem: float,\n):\n    proporcao = area_box / area_imagem\n\n    if proporcao < LIMITE_AREA_MEDIA:\n        return "pequeno", proporcao\n\n    if proporcao < LIMITE_AREA_GRANDE:\n        return "medio", proporcao\n\n    return "grande", proporcao\n\n\ndef classificar_posicao_vertical(\n    y2: float,\n    altura_imagem: int,\n):\n    proporcao_vertical = y2 / altura_imagem\n\n    if proporcao_vertical < LIMITE_VERTICAL_SUPERIOR:\n        return "superior", proporcao_vertical\n\n    if proporcao_vertical < LIMITE_VERTICAL_INFERIOR:\n        return "intermediaria", proporcao_vertical\n\n    return "inferior", proporcao_vertical\n\ndef classificar_risco_v1(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n):\n    if classe in CLASSES_IGNORADAS:\n        return {\n            "pontuacao": 0,\n            "nivel_risco": "ignorado",\n            "motivo": "classe sem relevância direta para locomoção",\n        }\n\n    if classe in CLASSES_CONTEXTUAIS:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "contextual",\n            "motivo": "classe exige informação adicional de contexto",\n        }\n\n    if classe not in PESO_OBJETO:\n        return {\n            "pontuacao": None,\n            "nivel_risco": "nao_mapeado",\n            "motivo": "classe não configurada",\n        }\n\n    pontuacao = (\n        PESO_OBJETO[classe]\n        + PESO_POSICAO[posicao]\n        + PESO_TAMANHO[tamanho]\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "motivo": (\n            f"{classe} + posição {posicao} + tamanho {tamanho}"\n        ),\n    }\n\ndef classificar_risco_v2(\n    classe: str,\n    posicao: str,\n    tamanho: str,\n    posicao_vertical: str,\n):\n    avaliacao_v1 = classificar_risco_v1(\n        classe=classe,\n        posicao=posicao,\n        tamanho=tamanho,\n    )\n\n    if avaliacao_v1["nivel_risco"] in {\n        "ignorado",\n        "contextual",\n        "nao_mapeado",\n    }:\n        return {\n            "pontuacao": avaliacao_v1["pontuacao"],\n            "nivel_risco": avaliacao_v1["nivel_risco"],\n            "ajuste_vertical": 0,\n            "motivo": avaliacao_v1["motivo"],\n        }\n\n    ajuste_vertical = AJUSTE_VERTICAL[posicao_vertical]\n\n    pontuacao = max(\n        1,\n        avaliacao_v1["pontuacao"] + ajuste_vertical,\n    )\n\n    if pontuacao <= 4:\n        nivel = "baixo"\n    elif pontuacao <= 6:\n        nivel = "medio"\n    else:\n        nivel = "alto"\n\n    return {\n        "pontuacao": pontuacao,\n        "nivel_risco": nivel,\n        "ajuste_vertical": ajuste_vertical,\n        "motivo": (\n            f"{classe} + posição {posicao} "\n            f"+ tamanho {tamanho} "\n            f"+ região vertical {posicao_vertical}"\n        ),\n    }\n', 'src/validacao_alertas.py': '"""Compara detectores na validação Home, com a V2 e os parâmetros do 05d."""\nfrom __future__ import annotations\n\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .politica_alertas import avaliar_bbox\n\nIMGSZ = 640\nCONF_ALERTA = 0.25\nIOU_NMS = 0.70\nIOU_CORRESPONDENCIA = 0.50\nMAX_DETECCOES = 300\n\n\ndef ler_objetos_yolo(caminho, largura, altura, nomes):\n    objetos = []\n    for n, linha in enumerate(Path(caminho).read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not linha.strip():\n            continue\n        partes = linha.split()\n        if len(partes) != 5:\n            raise ValueError(f"Label inválido: {caminho}, linha {n}")\n        valores = [float(p) for p in partes]\n        if not all(math.isfinite(v) for v in valores):\n            raise ValueError(f"Label não finito: {caminho}, linha {n}")\n        id_float, xc, yc, w, h = valores\n        c = int(id_float)\n        bordas = [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]\n        if id_float != c or c not in nomes or w <= 0 or h <= 0 or min(bordas) < -0.0001 or max(bordas) > 1.0001:\n            raise ValueError(f"Classe ou caixa inválida: {caminho}, linha {n}")\n        x1, y1, x2, y2 = np.clip(bordas, 0, 1)\n        objetos.append({"classe_id": c, "classe": nomes[c],\n                        "bbox": [float(x1 * largura), float(y1 * altura),\n                                 float(x2 * largura), float(y2 * altura)]})\n    return objetos\n\n\ndef carregar_home_val(dataset, nomes):\n    dataset = Path(dataset)\n    raiz_imagens = dataset / "images/val"\n    registros = []\n    for imagem in sorted(raiz_imagens.rglob("*")):\n        if not imagem.is_file() or imagem.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:\n            continue\n        relativo = imagem.relative_to(raiz_imagens)\n        if relativo.parts[0] == "sunrgbd":\n            continue\n        label = dataset / "labels/val" / relativo.with_suffix(".txt")\n        with Image.open(imagem) as im:\n            largura, altura = im.size\n            im.verify()\n        registros.append({"imagem": imagem, "nome": relativo.as_posix(),\n                          "largura": largura, "altura": altura,\n                          "gt": ler_objetos_yolo(label, largura, altura, nomes)})\n    if not registros:\n        raise ValueError("Validação Home vazia.")\n    return registros\n\n\ndef iou_matriz(gt, pred):\n    if not gt or not pred:\n        return np.zeros((len(gt), len(pred)), dtype=float)\n    g = np.asarray([o["bbox"] for o in gt], dtype=float)\n    p = np.asarray([o["bbox"] for o in pred], dtype=float)\n    inicio = np.maximum(g[:, None, :2], p[None, :, :2])\n    fim = np.minimum(g[:, None, 2:], p[None, :, 2:])\n    inter = np.prod(np.maximum(0.0, fim - inicio), axis=2)\n    ag = np.prod(np.maximum(0.0, g[:, 2:] - g[:, :2]), axis=1)\n    ap = np.prod(np.maximum(0.0, p[:, 2:] - p[:, :2]), axis=1)\n    uniao = ag[:, None] + ap[None, :] - inter\n    return np.divide(inter, uniao, out=np.zeros_like(inter), where=uniao > 0)\n\n\ndef corresponder_objetos(gt, pred):\n    matriz = iou_matriz(gt, pred)\n    candidatos = []\n    for gi, g in enumerate(gt):\n        for pi, p in enumerate(pred):\n            if g["classe_id"] == p["classe_id"] and matriz[gi, pi] >= IOU_CORRESPONDENCIA:\n                candidatos.append((float(matriz[gi, pi]), gi, pi))\n    candidatos.sort(key=lambda item: (-item[0], item[1], item[2]))\n    usados_g, usados_p, pares = set(), set(), []\n    for iou, gi, pi in candidatos:\n        if gi not in usados_g and pi not in usados_p:\n            pares.append((gi, pi, iou))\n            usados_g.add(gi)\n            usados_p.add(pi)\n    return pares, [i for i in range(len(gt)) if i not in usados_g], [i for i in range(len(pred)) if i not in usados_p]\n\n\ndef contagem_eventos(gt, pred, largura, altura):\n    """Um FN/FP pode decorrer de detecção ausente/extra ou mudança de prioridade."""\n    g = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in gt]\n    p = [{**o, **avaliar_bbox(o["classe"], o["bbox"], largura, altura)} for o in pred]\n    pares, faltantes, extras = corresponder_objetos(g, p)\n    eventos = [(g[gi], p[pi]) for gi, pi, _ in pares]\n    eventos += [(g[gi], None) for gi in faltantes]\n    eventos += [(None, p[pi]) for pi in extras]\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for ref, prev in eventos:\n        a = bool(ref and ref["prioridade"] in {"medio", "alto"})\n        b = bool(prev and prev["prioridade"] in {"medio", "alto"})\n        contagem["tp"] += int(a and b)\n        contagem["fn"] += int(a and not b)\n        contagem["fp"] += int(not a and b)\n        contagem["alta_sem_alerta"] += int(bool(ref and ref["prioridade"] == "alto" and not b))\n    return contagem\n\n\ndef avaliar_alertas(modelo, registros, nomes, device=0):\n    modelo_nomes = {int(k): str(v) for k, v in modelo.names.items()}\n    if modelo_nomes != nomes:\n        raise ValueError("Mapa de classes do modelo difere do projeto.")\n    contagem = {"tp": 0, "fn": 0, "fp": 0, "alta_sem_alerta": 0}\n    for numero, r in enumerate(registros, 1):\n        resultado = modelo.predict(str(r["imagem"]), conf=CONF_ALERTA, imgsz=IMGSZ,\n                                   iou=IOU_NMS, max_det=MAX_DETECCOES,\n                                   device=device, verbose=False, save=False)[0]\n        caixas = resultado.boxes\n        pred = [] if caixas is None else [\n            {"classe_id": int(c), "classe": nomes[int(c)], "bbox": [float(v) for v in b]}\n            for c, b in zip(caixas.cls.cpu().numpy(), caixas.xyxy.cpu().numpy())]\n        eventos = contagem_eventos(r["gt"], pred, r["largura"], r["altura"])\n        for k, v in eventos.items():\n            contagem[k] += v\n        if numero % 100 == 0:\n            print(f"Validação dos alertas: {numero}/{len(registros)}")\n    tp, fn, fp = contagem["tp"], contagem["fn"], contagem["fp"]\n    dividir = lambda n, d: n / d if d else None\n    return {"imagens": len(registros), **contagem,\n            "precision_alertas": dividir(tp, tp + fp),\n            "recall_alertas": dividir(tp, tp + fn),\n            "f1_alertas": dividir(2 * tp, 2 * tp + fp + fn),\n            "referencia": "V2 fixa nas anotações; não valida risco físico independentemente",\n            "conf": CONF_ALERTA, "iou_nms": IOU_NMS,\n            "iou_correspondencia": IOU_CORRESPONDENCIA}\n'}
CODIGO_SHA256={'conf14_data.py': 'f73232836278c08edf0f955ae8fe50a33cb1d4eabe2b2cc35a5f410f095bae66', 'conf14_evaluation.py': '3e64c612ffcd7cb75fdf05707affc45cafa91acd1882ab311425c2752ed01656', 'diag11_core.py': '29e7c52fced44ab4898edfbc46477176587f290d2fbdacad4de5f595d4187670', 'exp12_protocol.py': '257d09b0b2aafe94da28bdcb59859778ffb31e25601db75d1058092b943b5759', 'final20_data.py': '0c31947fca9018c69fbd6c1933fcb51a44158ac7e7a6f709ed38638c4e5b8538', 'final20_drive.py': '25300e50e08260a7a0c0601fb65941605d042c97ec23cbb46811e4e0900f14eb', 'final20_eval.py': '3555c648a33ea58e128b66048c34b2677e3c4e53b5880e23d11cefe2b287d42d', 'final20_io.py': 'd4fee2cffadd2578b004b2bd9f33c3caa79a97afe2f26b2ddd658a1b0a230b17', 'final20_report.py': 'e6e2370810a971db780b3df6e093ed187666244ef4fa3adf24179ae1e0b32ce5', 'final20_stats.py': '8120d948239baea268a10684b98049536d9c26bb2a00eb0ecf639b93cd7bc809', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'src/politica_alertas.py': '1aa6b13cb68d8f30a2f388c3577a61a12dcac4c604581877091d3fbdc6042845', 'src/validacao_alertas.py': '1da5dff21a6c6f6ec53d041247ce18d012008c4b26023b52c3fd2745f4c4105d'}
PASTA_CODIGO=PASTA_LOCAL/'codigo';PASTA_CODIGO.mkdir(exist_ok=True)
for nome,fonte in MODULOS_EMBUTIDOS.items():
    if hashlib.sha256(fonte.encode()).hexdigest()!=CODIGO_SHA256[nome]:raise RuntimeError('Código embutido alterado: '+nome)
    compile(fonte,nome,'exec')
    for raiz in [PASTA_CODIGO,PASTA_BOOT/'codigo']:
        destino=raiz/nome;destino.parent.mkdir(parents=True,exist_ok=True);destino.write_text(fonte,encoding='utf-8')
sys.path.insert(0,str(PASTA_CODIGO))
import diag11_core as core
import final20_data as dados20
import final20_eval as avaliacao20
import final20_stats as estatistica20
import final20_report as relatorio20
import final20_io as arquivos20
from final20_drive import Drive
from src import politica_alertas as politica
from src import validacao_alertas as associacao
if dados20.digest(PROTOCOLO_CONGELADO)!=EXPECTED['protocolo_sha256']:
    raise RuntimeError('O plano embutido difere do congelado no 19.')
if core.sha256_file(politica.__file__)!=PROTOCOLO_CONGELADO['policy_sha256'] or core.sha256_file(associacao.__file__)!=PROTOCOLO_CONGELADO['matcher_sha256'] or associacao.IOU_CORRESPONDENCIA!=PROTOCOLO_CONGELADO['match_iou']:
    raise RuntimeError('Política ou associação diferentes do protocolo.')
def progresso(etapa):
    global ULTIMA_ETAPA
    agora=time.monotonic();TEMPOS.append({'etapa':etapa,'segundos':agora-ULTIMA_ETAPA,'acumulado_segundos':agora-INICIO});ULTIMA_ETAPA=agora
    core.save_json(PASTA_RESULTADOS/'tempos_execucao_20.json',{'exclui_pip':True,'exclui_zip_e_envio_final':True,'etapas':TEMPOS})
    print(f'{etapa} | {(agora-INICIO)/60:.1f} min acumulados',flush=True)
def salvar_checkpoint():
    return arquivos20.backup(DRIVE,PASTA_RESULTADOS,'resultados/'+NOME_EXPERIMENTO)
def registrar_falha(exc,etapa):
    pacote=arquivos20.failure(PASTA_RESULTADOS,exc,etapa,DRIVE,'resultados/'+NOME_EXPERIMENTO)
    print('Execução interrompida. Previsões concluídas foram preservadas. Pacote:',pacote)
    display(FileLink(str(pacote)))
progresso('Ambiente, código e protocolo embutido conferidos')


## 2. Autenticar o protocolo 19 e os dois pesos

Confere o pacote 19 recebido, o plano registrado antes da reserva e o SHA-256 dos
dois pesos. Só depois dessas verificações a reserva poderá ser extraída. A pasta
de resultados é determinada pelo código, pelo protocolo e pelo ambiente.

Se uma execução anterior foi interrompida, os arquivos salvos no Drive são
recuperados automaticamente quando não existem resultados locais desta execução.


In [ ]:
try:
    DRIVE=Drive.from_kaggle()
    if shutil.disk_usage(PASTA_LOCAL).free<4*1024**3:raise RuntimeError('São necessários 4 GiB livres para as entradas e a reserva.')
    ENTRADAS=PASTA_LOCAL/'entradas'
    PACOTE_19=DRIVE.fetch(EXPECTED['pasta_19']+'/pacote_consolidacao_19.zip',ENTRADAS/'pacote_19.zip',EXPECTED['pacote_19'])
    PLANO,DESENVOLVIMENTO=dados20.source19(PACOTE_19,EXPECTED,PROTOCOLO_CONGELADO,PASTA_BOOT)
    PESOS={}
    for modelo in dados20.MODELS:
        PESOS[modelo]=DRIVE.fetch(EXPECTED['weights'][modelo],ENTRADAS/(modelo+'.pt'),PLANO['weight_sha256'][modelo])
        if core.sha256_file(PESOS[modelo])!=PLANO['weight_sha256'][modelo]:raise RuntimeError('Peso não corresponde ao plano: '+modelo)
    CONTEXTO={'versao':'20-avaliacao-final-1.0.0','pacote_19_sha256':EXPECTED['pacote_19'],
        'protocolo_sha256':dados20.digest(PLANO),'codigo_sha256':CODIGO_SHA256,'ambiente':AMBIENTE}
    CONTEXT_KEY=dados20.digest(CONTEXTO);NOME_EXPERIMENTO='avaliacao20_'+CONTEXT_KEY[:24]
    PASTA_RESULTADOS=Path('/kaggle/working/resultados_tcc')/NOME_EXPERIMENTO
    PASTA_RESULTADOS.mkdir(parents=True,exist_ok=True)
    if not (PASTA_RESULTADOS/'configuracao_avaliacao_20.json').is_file():
        DRIVE.restore('resultados/'+NOME_EXPERIMENTO,PASTA_RESULTADOS)
    configuracao=PASTA_RESULTADOS/'configuracao_avaliacao_20.json'
    if configuracao.is_file() and json.loads(configuracao.read_text())['context_key']!=CONTEXT_KEY:
        raise RuntimeError('Resultado anterior pertence a outro contexto.')
    shutil.copytree(PASTA_BOOT,PASTA_RESULTADOS,dirs_exist_ok=True)
    core.save_json(configuracao,{'context_key':CONTEXT_KEY,**CONTEXTO,'protocolo':PLANO,'expected':EXPECTED})
    if not (PASTA_RESULTADOS/'registro_abertura_reserva_20.json').is_file():
        core.save_json(PASTA_RESULTADOS/'registro_abertura_reserva_20.json',{
            'registrado_em_utc':arquivos20.now(),'protocolo_fonte_fixado_em_utc':json.loads((PASTA_RESULTADOS/'fontes19/registro_protocolo_final.json').read_text())['fixado_em_utc'],
            'protocolo_sha256':dados20.digest(PLANO),'pesos_sha256':PLANO['weight_sha256'],
            'reserva_extraida_antes_deste_registro':False,'execucao_autorizada_da_avaliacao_final':True})
    arquivos20.state(PASTA_RESULTADOS,concluido=False,context_key=CONTEXT_KEY,etapa='protocolo e pesos conferidos',
        treinamento_executado=False,modelo_substituido=False)
    progresso('Protocolo 19 e pesos autenticados; contexto de retomada preparado')
    salvar_checkpoint()
except Exception as exc:
    registrar_falha(exc,'protocolo e pesos');raise


## 3. Abrir a reserva final e formar os grupos

Baixa o ZIP preservado e extrai somente imagens, labels e cobertura dos **300 IDs
reserva_futura**. Não substitui ou remove imagens. Verifica os bytes e mantém a
cobertura parcial original. As referências de desenvolvimento são comparadas
por metadados, sem executar nova inferência nelas.

Os grupos usam os pHashes congelados no manifesto original: bytes iguais ou
distância de Hamming até 4 unem imagens, inclusive por ligação transitiva.
Esses mesmos grupos serão usados em todas as comparações.


In [ ]:
try:
    ZIP_RESERVA=DRIVE.fetch(PLANO['dataset_location'],ENTRADAS/'validacao_preservada.zip',PLANO['archive_sha256'])
    REGISTROS,AUDITORIA_EXTRACAO=dados20.extract_reserve(ZIP_RESERVA,PLANO,DESENVOLVIMENTO,PASTA_LOCAL/'reserva_extraida',PASTA_RESULTADOS)
    GRUPOS,MAPA_GRUPOS,RESUMO_GRUPOS=estatistica20.make_groups(REGISTROS,PASTA_RESULTADOS)
    arquivos20.state(PASTA_RESULTADOS,etapa='reserva extraída',reserva_extraida=True,imagens_reserva=len(REGISTROS))
    print('Imagens:',len(REGISTROS),'| Grupos:',len(GRUPOS))
    if AUDITORIA_EXTRACAO['pares_semelhanca_phash_com_desenvolvimento']:
        print('Semelhanças pHash com desenvolvimento foram registradas para discussão. Nenhuma imagem foi substituída.')
    progresso('300 imagens da reserva conferidas e grupos fixados')
    salvar_checkpoint()
except Exception as exc:
    registrar_falha(exc,'abertura da reserva');raise


## 4. Avaliar os dois modelos fixos

Executa **600 inferências no máximo** em uma execução sem interrupções: 300 por
modelo, uma imagem por chamada. Confiança de captura 0,001, entrada 640, NMS IoU
0,70, max_det 300, FP32 e batch 1. A avaliação usa o limiar fixo 0,25.

As previsões são gravadas atomicamente em lotes de 25 imagens. A cada 100 imagens
e ao terminar cada modelo, os arquivos são copiados para o Drive. A retomada
confere o contexto e reutiliza os lotes concluídos. Propostas com área zero ou
fora da imagem são diagnosticadas, preservando o comportamento dos avaliadores
anteriores; não há recorte ou filtro novo.

As mensagens usam todas as previsões relevantes. A pontuação usa somente classes
com cobertura de referência; classes sem cobertura não viram falsos positivos
ou negativos presumidos.


In [ ]:
try:
    AVALIACOES={}
    arquivos20.state(PASTA_RESULTADOS,etapa='inferência dos modelos fixos',inferencia_iniciada=True)
    for modelo in dados20.MODELS:
        AVALIACOES[modelo]=avaliacao20.run_model(PESOS[modelo],REGISTROS,modelo,AMBIENTE,DEVICE,PLANO,
            politica,associacao.corresponder_objetos,PASTA_RESULTADOS/'avaliacoes'/modelo,salvar_checkpoint)
        arquivos20.state(PASTA_RESULTADOS,etapa='modelo concluído: '+modelo,ultimo_modelo_concluido=modelo)
        progresso('Avaliação concluída: '+modelo)
        salvar_checkpoint()
    print('Dois modelos avaliados nas mesmas',len(REGISTROS),'imagens.')
except Exception as exc:
    registrar_falha(exc,'inferência e métricas');raise


## 5. Calcular os intervalos de confiança pareados

Usa **2.000 réplicas**, semente **19002026** e intervalos percentis de **95%**.
Cada réplica reamostra o mesmo número de grupos com reposição; as multiplicidades
são idênticas para ambos os modelos. São calculadas razões de contagens agregadas.

Os estimandos primários são o **recall de alertas do baseline** e a **diferença de
recall candidato 10 − baseline**. Os demais resultados são secundários. Réplicas
sem denominador são indefinidas e sua quantidade válida aparece nas tabelas.
As 12 classes são mantidas; suporte abaixo de 30 GT é sinalizado.


In [ ]:
try:
    INTERVALOS,RESUMO_BOOTSTRAP=estatistica20.bootstrap(REGISTROS,AVALIACOES,GRUPOS,PLANO,PASTA_RESULTADOS)
    progresso('Intervalos de confiança e comparação pareada concluídos')
    salvar_checkpoint()
except Exception as exc:
    registrar_falha(exc,'intervalos de confiança');raise


## 6. Gerar o relatório e encerrar a fase experimental

Produz tabelas, resultados por classe, incerteza e texto-base. Os dados das duas
validações de desenvolvimento ficam separados da reserva final. Não aplica os
limites absolutos de 1.000 imagens a este conjunto de 300 e não escolhe um modelo
novo com base na reserva.

O ZIP contém previsões, GT, grupos, réplicas estatísticas, código e protocolo,
sem pesos, imagens ou credenciais. Se o envio ao Drive falhar, o resultado local
continua disponível para download. A última célula pode ser executada novamente
para repetir somente a geração e o envio do pacote.


In [ ]:
try:
    RESUMO=relatorio20.build(REGISTROS,AVALIACOES,INTERVALOS,RESUMO_BOOTSTRAP,RESUMO_GRUPOS,AUDITORIA_EXTRACAO,
        PLANO,EXPECTED,CONTEXT_KEY,PASTA_RESULTADOS)
    progresso('Relatório final e texto-base concluídos')
    arquivos20.state(PASTA_RESULTADOS,concluido=True,etapa='avaliação final concluída',reserva_inferida=True,
        imagens_reserva=len(REGISTROS),modelos_concluidos=list(dados20.MODELS),pesos_ou_limiares_ajustados=False)
    (PASTA_RESULTADOS/'execucao_interrompida_20.json').unlink(missing_ok=True)
    arquivos20.manifest(PASTA_RESULTADOS)
    PACOTE_FINAL=arquivos20.package(PASTA_RESULTADOS)
    SALVO_DRIVE=salvar_checkpoint()
    core.save_json(PASTA_RESULTADOS/'salvamento_drive_20.json',{'sucesso':SALVO_DRIVE,'verificado_em_utc':arquivos20.now(),
        'observacao':'O pacote contém os resultados científicos completos; este recibo local registra o envio final.'})
    print('\nAVALIAÇÃO FINAL CONCLUÍDA — fase experimental encerrada.')
    print('Resultados em:',PASTA_RESULTADOS)
    print('Arquivo de resultados:',PACOTE_FINAL.name)
    if not SALVO_DRIVE:print('Baixe o ZIP pelo link abaixo. Você pode repetir esta última célula para tentar o envio novamente.')
    for nome in ['pacote_avaliacao_final_20.zip','relatorio_avaliacao_final_20.html','comparacao_final_20.csv',
                 'resultados_por_classe_20.csv','texto_base_resultados_finais_20.md']:
        display(FileLink(str(PASTA_RESULTADOS/nome)))
except Exception as exc:
    registrar_falha(exc,'relatório e pacote');raise
